In [15]:
import csv
import os
from fb2reader import fb2book
from bs4 import BeautifulSoup

book = fb2book('/Users/kseniazavyalova/Desktop/книги/A Feast for Crows - A Song of Ice and Fire, Book 4 (George R. R. Martin) (z-library.sk, 1lib.sk, z-lib.sk).fb2')

title = book.get_title() or "Нет названия"
authors_data = book.get_authors() or []
authors_str = ", ".join(
    a.get("full_name") or f"{a.get('first_name', '')} {a.get('last_name', '')}".strip()
    for a in authors_data
) or "Нет автора"
genre_str = ", ".join(book.get_tags() or []) or "Нет жанра"

raw_body = book.get_body()
html_content = "".join(raw_body) if isinstance(raw_body, list) else str(raw_body)

soup = BeautifulSoup(html_content, "html.parser")
clean_text = soup.get_text(separator="\n\n", strip=True)

# Явно указываем путь, куда разрешена запись
output_path = "/Users/kseniazavyalova/Desktop/book_data.csv"

# Убеждаемся, что родительская директория существует
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, mode="w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["Title", "Authors", "Genre", "Text"])
    writer.writeheader()
    writer.writerow({
        "Title": title,
        "Authors": authors_str,
        "Genre": genre_str,
        "Text": clean_text
    })

print(f"Файл успешно сохранён: {output_path}")
print(f"Размер записанного текста: {len(clean_text)} символов")

Файл успешно сохранён: /Users/kseniazavyalova/Desktop/book_data.csv
Размер записанного текста: 1713576 символов


In [18]:
import csv
from pathlib import Path
import tempfile
import os
import chardet
from fb2reader import fb2book
from bs4 import BeautifulSoup

input_dir = Path("/Users/kseniazavyalova/Desktop/книги/")
output_csv = Path("/Users/kseniazavyalova/Desktop/all_books_data.csv")

# Собираем все FB2 файлы
fb2_files = sorted(list(input_dir.glob("*.fb2")) + list(input_dir.glob("*.fb2.zip")))

with open(output_csv, mode="w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["Title", "Authors", "Genre", "Text"])
    writer.writeheader()

    processed = 0
    skipped = 0

    for idx, file_path in enumerate(fb2_files, 1):
        tmp_path = None
        try:
            print(f"[{idx}] Обработка: {file_path.name}")
            
            # Попытка открыть напрямую
            try:
                book = fb2book(str(file_path))
            except Exception as e:
                if "decode" in str(e).lower() or "unicode" in str(e).lower():
                    print(f"  ⚠️ Несоответствие кодировки. Определяю и конвертирую...")
                    with open(file_path, 'rb') as f:
                        raw_bytes = f.read()
                    
                    detected = chardet.detect(raw_bytes)
                    encoding = detected['encoding'] or 'windows-1251'
                    decoded_text = raw_bytes.decode(encoding, errors='replace')
                    
                    # Сохраняем исправленную версию во временный UTF-8 файл
                    tmp = tempfile.NamedTemporaryFile(mode='w', suffix='.fb2', delete=False, encoding='utf-8')
                    tmp.write(decoded_text)
                    tmp.close()
                    tmp_path = tmp.name
                    
                    book = fb2book(tmp_path)
                else:
                    raise

            # 1. Метаданные
            title = book.get_title() or "Без названия"
            authors_data = book.get_authors() or []
            authors_str = ", ".join(
                a.get("full_name") or f"{a.get('first_name', '')} {a.get('last_name', '')}".strip()
                for a in authors_data
            ) or "Нет автора"
            genre_str = ", ".join(book.get_tags() or []) or "Без жанра"

            # 2. Очистка текста
            raw_body = book.get_body()
            if not raw_body:
                raise ValueError("Текст книги не найден в файле")

            html_content = "".join(raw_body) if isinstance(raw_body, list) else str(raw_body)
            soup = BeautifulSoup(html_content, "html.parser")
            clean_text = soup.get_text(separator="\n\n", strip=True)

            # 3. Запись в CSV
            writer.writerow({
                "Title": title,
                "Authors": authors_str,
                "Genre": genre_str,
                "Text": clean_text
            })
            processed += 1

        except Exception as e:
            print(f"  ❌ Пропущен {file_path.name}: {e}")
            skipped += 1
        finally:
            # Удаляем временный файл, если он был создан
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)

print(f"\n✅ Готово. Успешно: {processed}, Ошибки: {skipped}")
print(f"📄 Файл сохранён: {output_csv}")

[1] Обработка: A Feast for Crows - A Song of Ice and Fire, Book 4 (George R. R. Martin) (z-library.sk, 1lib.sk, z-lib.sk).fb2
[2] Обработка: A Warriors Journey - Ergoth Trilogy, Volume 1 (Paul B. Thompson, Tonya C. Cook) (z-library.sk, 1lib.sk, z-lib.sk).fb2
  ⚠️ Несоответствие кодировки. Определяю и конвертирую...
[3] Обработка: BRIDE (Ali Hazelwood) (z-library.sk, 1lib.sk, z-lib.sk).fb2
[4] Обработка: Bouen_Bob-the-Cat_1_A-Street-Cat-Named-Bob-How-One-Man-and-His-Cat-Found-Hope-on-the-Streets_RuLit_Me.fb2
[5] Обработка: City of Glass (Clare Cassandra) (z-library.sk, 1lib.sk, z-lib.sk).fb2
  ⚠️ Несоответствие кодировки. Определяю и конвертирую...
[6] Обработка: Deas_The-Thief-Taker-s-Blade_RuLit_Me.fb2
  ⚠️ Несоответствие кодировки. Определяю и конвертирую...
[7] Обработка: Fourth Wing.fb2
[8] Обработка: Gideon the Ninth (The Locked Tomb Tetralogy 1) (Tamsyn Muir) (z-library.sk, 1lib.sk, z-lib.sk).fb2
[9] Обработка: House of Flame and Shadow (Sarah J. Maas) (z-library.sk, 1lib.sk, z-l

In [19]:
import pandas as pd

df = pd.read_csv("/Users/kseniazavyalova/Desktop/all_books_data.csv")

# Добавляем колонки с метриками
df["char_count"] = df["Text"].str.len()
df["word_count"] = df["Text"].str.split().str.len()

# Вывод статистики
print("📈 Статистика по длинам текстов:\n")
print(df[["Title", "char_count", "word_count"]].sort_values("char_count", ascending=False))

print(f"\nСводка:")
print(f"Книг: {len(df)}")
print(f"Символов всего: {df['char_count'].sum():,}")
print(f"Слов всего: {df['word_count'].sum():,}")
print(f"Среднее: {df['char_count'].mean():,.0f} символов / {df['word_count'].mean():,.0f} слов")
print(f"Мин/Макс: {df['char_count'].min():,} – {df['char_count'].max():,} символов")

📈 Статистика по длинам текстов:

                                                Title  char_count  word_count
0                                   A Feast for Crows     1713576      314961
8            CRESCENT: CITY HOUSE OF FLAME AND SHADOW     1409127      249113
27                               The Will of the Many     1365151      240262
31  Zodiac Academy 4: Shadow Princess: An Academy ...     1179606      222930
6                                         Fourth Wing     1021396      185789
22                                        CHAPTER ONE      918285      166704
4                                       City of Glass      839589      150832
7                                    Gideon the Ninth      810634      141115
26                    Tamora Pierce - The Will of the      738854      133557
24                                    The Serpent Sea      687955      122804
1                                 A Warrior_s Journey      687792      119853
28                             

In [22]:
import pandas as pd
import re
import spacy
from transformers import pipeline
from flair.models import SequenceTagger
from flair.data import Sentence
from collections import defaultdict
import warnings
import os

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. НАСТРОЙКИ И ЗАГРУЗКА МОДЕЛЕЙ (CPU)
# ==============================================================================
DEVICE = -1  # -1 означает CPU
CHUNK_SIZE = 2500

print("[INFO] Загрузка моделей NER...")
SPACY_NER = spacy.load("en_core_web_trf")
HF_NER = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=DEVICE)
FLAIR_NER = SequenceTagger.load("ner")
print("[INFO] Модели загружены.")

STOP_NAMES = {
    'god', 'christ', 'jesus', 'satan', 'devil', 'hell', 'heaven', 'lord',
    'england', 'london', 'france', 'america', 'europe', 'world', 'earth',
    'time', 'day', 'night', 'year', 'month', 'week', 'monday', 'sunday',
    'mr', 'mrs', 'miss', 'ms', 'dr', 'sir', 'madam', 'lady', 'captain',
    'colonel', 'general', 'reverend', 'father', 'sister', 'brother', 'king',
    'queen', 'prince', 'princess', 'duke', 'duchess', 'earl', 'count',
    'chapter', 'volume', 'book', 'part', 'scene', 'act'
}

# ==============================================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==============================================================================
def normalize_name(name: str) -> str:
    name = re.sub(r'[^a-zA-Z\s]', '', name).lower().strip()
    name = re.sub(r'^(mr|mrs|miss|ms|dr|sir|madam|lady|lord|captain|colonel|general|reverend|father|sister)\s+', '', name)
    name = re.sub(r'\b[a-z]\.\s*', '', name)
    return ' '.join(name.split())

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    words = text.split()
    chunks = []
    current_chunk = []
    current_len = 0
    for word in words:
        current_chunk.append(word)
        current_len += len(word) + 1
        if current_len >= chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_len = 0
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

# ==============================================================================
# 3. ЭКСТРАКТОРЫ
# ==============================================================================
def extract_spacy_trf(text: str) -> list[dict]:
    doc = SPACY_NER(text)
    return [{"name": normalize_name(ent.text), "conf": float(getattr(ent._.trf_data, 'tok2tensor', 0.8)), "method": "spacy_trf"}
            for ent in doc.ents if ent.label_ == "PERSON" and normalize_name(ent.text) not in STOP_NAMES]

def extract_hf_bert(text: str) -> list[dict]:
    results = HF_NER(text)
    cleaned = []
    for r in results:
        if r.get("entity_group") == "PER":
            word = r["word"].replace(" ##", "")
            norm = normalize_name(word)
            if norm not in STOP_NAMES:
                cleaned.append({"name": norm, "conf": float(r["score"]), "method": "hf_bert"})
    return cleaned

def extract_flair_ner(text: str) -> list[dict]:
    candidates = []
    for chunk in chunk_text(text):
        if not chunk.strip(): continue
        flair_sent = Sentence(chunk)
        FLAIR_NER.predict(flair_sent)
        for entity in flair_sent.get_spans('ner'):
            if entity.tag == 'PER':
                norm = normalize_name(entity.text)
                if norm not in STOP_NAMES:
                    candidates.append({"name": norm, "conf": float(entity.score), "method": "flair"})
    return candidates

def extract_statistical_fallback(text: str, min_freq: int = 3) -> list[dict]:
    doc = SPACY_NER(text)
    propn_counts = defaultdict(int)
    title_pattern = re.compile(r'\b(Mr|Mrs|Miss|Ms|Sir|Lady|Lord|Captain|Colonel|General|Reverend|Father|Sister)\.\?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)')

    for token in doc:
        if token.pos_ == "PROPN" and len(token.text) > 2:
            norm = normalize_name(token.text)
            if norm and norm not in STOP_NAMES:
                propn_counts[norm] += 1

    candidates = [{"name": name, "conf": 0.50 + 0.05 * min(freq, 6), "method": "stat_propn"}
                  for name, freq in propn_counts.items() if freq >= min_freq]

    for match in title_pattern.finditer(text):
        norm = normalize_name(match.group(2))
        if norm and norm not in STOP_NAMES:
            candidates.append({"name": norm, "conf": 0.75, "method": "stat_title"})
    return candidates

# ==============================================================================
# 4. ЗАПУСК НА ПЕРВОЙ КНИГЕ
# ==============================================================================
def run_ensemble_first_book(csv_path: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    
    df_books = pd.read_csv(csv_path)
    first_book = df_books.iloc[0]
    book_title = str(first_book.get('Title', 'Unknown'))
    text = str(first_book.get('Text', ''))

    print(f"[INFO] Книга: {book_title}")
    print(f"[INFO] Длина текста: {len(text):,} символов")
    print("[INFO] Запуск извлечения... (ожидание на CPU)")

    results = []
    results.extend(extract_spacy_trf(text))
    print("  [OK] spaCy_trf")
    results.extend(extract_hf_bert(text))
    print("  [OK] hf_bert")
    results.extend(extract_flair_ner(text))
    print("  [OK] flair")
    results.extend(extract_statistical_fallback(text))
    print("  [OK] statistical_fallback")

    if not results:
        print("[WARN] Имена не найдены.")
        return

    results_df = pd.DataFrame(results)

    # Таблица 1: Список по каждому методу
    method_lists = []
    for method in results_df['method'].unique():
        subset = results_df[results_df['method'] == method]
        unique_names = sorted(subset['name'].unique())
        method_lists.append({
            "Method": method,
            "Unique_Count": len(unique_names),
            "Extracted_Names": ", ".join(unique_names)
        })
    summary_df = pd.DataFrame(method_lists)

    # Таблица 2: Матрица согласования (Имя x Метод)
    pivot_data = defaultdict(dict)
    for _, row in results_df.iterrows():
        pivot_data[row['name']][row['method']] = row['conf']

    cross_df = pd.DataFrame.from_dict(pivot_data, orient='index').fillna(0)
    cross_df.index.name = "Normalized_Name"
    cross_df = cross_df.reindex(columns=["spacy_trf", "hf_bert", "flair", "stat_propn", "stat_title"], fill_value=0)

    # Сохранение
    summary_path = os.path.join(output_dir, "ensemble_by_method.csv")
    cross_path = os.path.join(output_dir, "ensemble_cross_matrix.csv")
    
    summary_df.to_csv(summary_path, index=False)
    cross_df.to_csv(cross_path)

    print(f"\n[OK] Сводка по методам сохранена: {summary_path}")
    print(f"[OK] Матрица согласования сохранена: {cross_path}")
    print("\nСводка по методам:")
    for _, row in summary_df.iterrows():
        print(f"- {row['Method']}: {row['Unique_Count']} имён")
        
    return summary_df, cross_df

if __name__ == "__main__":
    CSV_INPUT = "/Users/kseniazavyalova/Desktop/all_books_data.csv"
    OUTPUT_DIR = "/Users/kseniazavyalova/Desktop/ensemble_results"
    
    run_ensemble_first_book(CSV_INPUT, OUTPUT_DIR)

ImportError: cannot import name 'is_tf_available' from 'transformers.utils' (/Users/kseniazavyalova/.pyenv/versions/3.11.0/lib/python3.11/site-packages/transformers/utils/__init__.py)

In [21]:
!pip install flair

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 2.8 MB/s eta 0:00:0000:0100:01
  DEPRECATION: Building 'pptree' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pptree'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for pptree: filename=pptree-3.1-py3-none-any.whl size=4607 sha256=94c1f0e6a382083e7e9635bfa5a0d5f96998eb30

In [1]:
import pandas as pd
import re
import spacy
from transformers import pipeline
from collections import defaultdict
import warnings
import os

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. НАСТРОЙКИ И ЗАГРУЗКА МОДЕЛЕЙ (CPU)
# ==============================================================================
DEVICE = -1  # -1 означает CPU
CHUNK_SIZE = 2000  # символов на чанк для снижения нагрузки на память

print("[INFO] Загрузка моделей NER...")
try:
    SPACY_NER = spacy.load("en_core_web_trf")
    HF_BERT = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=DEVICE)
    HF_ROBERTA = pipeline("ner", model="Jean-Baptiste/roberta-large-ner-english", aggregation_strategy="simple", device=DEVICE)
    print("[INFO] Модели успешно загружены.")
except Exception as e:
    print(f"[ERROR] Ошибка загрузки моделей: {e}")
    print("Выполните: pip install --upgrade transformers spacy-transformers")
    raise SystemExit

STOP_NAMES = {
    'god', 'christ', 'jesus', 'satan', 'devil', 'hell', 'heaven', 'lord',
    'england', 'london', 'france', 'america', 'europe', 'world', 'earth',
    'time', 'day', 'night', 'year', 'month', 'week', 'monday', 'sunday',
    'mr', 'mrs', 'miss', 'ms', 'dr', 'sir', 'madam', 'lady', 'captain',
    'colonel', 'general', 'reverend', 'father', 'sister', 'brother', 'king',
    'queen', 'prince', 'princess', 'duke', 'duchess', 'earl', 'count',
    'chapter', 'volume', 'book', 'part', 'scene', 'act'
}

# ==============================================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ==============================================================================
def normalize_name(name: str) -> str:
    name = re.sub(r'[^a-zA-Z\s]', '', name).lower().strip()
    name = re.sub(r'^(mr|mrs|miss|ms|dr|sir|madam|lady|lord|captain|colonel|general|reverend|father|sister)\s+', '', name)
    name = re.sub(r'\b[a-z]\.\s*', '', name)
    return ' '.join(name.split())

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    words = text.split()
    chunks, current, current_len = [], [], 0
    for word in words:
        current.append(word)
        current_len += len(word) + 1
        if current_len >= chunk_size:
            chunks.append(" ".join(current))
            current, current_len = [], 0
    if current:
        chunks.append(" ".join(current))
    return chunks

# ==============================================================================
# 3. ЭКСТРАКТОРЫ
# ==============================================================================
def extract_spacy_trf(text: str) -> list[dict]:
    doc = SPACY_NER(text)
    out = []
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            norm = normalize_name(ent.text)
            if norm not in STOP_NAMES:
                conf = float(getattr(ent._.trf_data, 'tok2tensor', 0.85)) if hasattr(ent._, 'trf_data') else 0.85
                out.append({"name": norm, "conf": conf, "method": "spacy_trf"})
    return out

def extract_hf_bert(text: str) -> list[dict]:
    out = []
    for chunk in chunk_text(text):
        if not chunk.strip(): continue
        for r in HF_BERT(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace(" ##", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "hf_bert"})
    return out

def extract_hf_roberta(text: str) -> list[dict]:
    out = []
    for chunk in chunk_text(text):
        if not chunk.strip(): continue
        for r in HF_ROBERTA(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace(" ##", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "hf_roberta"})
    return out

def extract_statistical_fallback(text: str, min_freq: int = 3) -> list[dict]:
    doc = SPACY_NER(text)
    propn_counts = defaultdict(int)
    title_pattern = re.compile(r'\b(Mr|Mrs|Miss|Ms|Sir|Lady|Lord|Captain|Colonel|General|Reverend|Father|Sister)\.\?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)')

    for token in doc:
        if token.pos_ == "PROPN" and len(token.text) > 2:
            norm = normalize_name(token.text)
            if norm and norm not in STOP_NAMES:
                propn_counts[norm] += 1

    candidates = [{"name": name, "conf": 0.50 + 0.05 * min(freq, 6), "method": "stat_propn"}
                  for name, freq in propn_counts.items() if freq >= min_freq]

    for match in title_pattern.finditer(text):
        norm = normalize_name(match.group(2))
        if norm and norm not in STOP_NAMES:
            candidates.append({"name": norm, "conf": 0.75, "method": "stat_title"})
    return candidates

# ==============================================================================
# 4. ЗАПУСК НА ПЕРВОЙ КНИГЕ
# ==============================================================================
def run_ensemble_first_book(csv_path: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    
    df_books = pd.read_csv(csv_path)
    if df_books.empty or 'Text' not in df_books.columns:
        print("[ERROR] В CSV нет колонки 'Text' или файл пуст.")
        return
        
    first_book = df_books.iloc[0]
    book_title = str(first_book.get('Title', 'Unknown_Book'))
    text = str(first_book.get('Text', ''))

    print(f"[INFO] Книга: {book_title}")
    print(f"[INFO] Длина текста: {len(text):,} символов")
    print("[INFO] Запуск извлечения... (ожидание на CPU)")

    raw_results = []
    raw_results.extend(extract_spacy_trf(text))
    print("  [OK] spaCy_trf")
    raw_results.extend(extract_hf_bert(text))
    print("  [OK] hf_bert")
    raw_results.extend(extract_hf_roberta(text))
    print("  [OK] hf_roberta")
    raw_results.extend(extract_statistical_fallback(text))
    print("  [OK] statistical_fallback")

    if not raw_results:
        print("[WARN] Имена не найдены.")
        return

    df_raw = pd.DataFrame(raw_results)
    
    # Группировка: метод -> список уникальных имён + средняя уверенность
    summary_rows = []
    for method in df_raw['method'].unique():
        subset = df_raw[df_raw['method'] == method]
        unique_names = sorted(subset['name'].unique())
        avg_conf = subset['conf'].mean()
        summary_rows.append({
            "Method": method,
            "Extracted_Name": ", ".join(unique_names),
            "Unique_Count": len(unique_names),
            "Avg_Confidence": round(avg_conf, 4)
        })
    
    df_summary = pd.DataFrame(summary_rows)
    output_path = os.path.join(output_dir, f"ner_extraction_{book_title.replace(' ', '_')}.csv")
    df_summary.to_csv(output_path, index=False)

    print(f"\n[OK] Таблица сохранена: {output_path}")
    print("\nРезультат по методам:")
    for _, row in df_summary.iterrows():
        print(f"- {row['Method']}: {row['Unique_Count']} имён (avg_conf={row['Avg_Confidence']})")
        
    return df_summary

if __name__ == "__main__":
    CSV_INPUT = "/Users/kseniazavyalova/Desktop/all_books_data.csv"
    OUTPUT_DIR = "/Users/kseniazavyalova/Desktop/ner_results"
    
    # Если возникает ошибка загрузки, выполните в терминале:
    # pip install --upgrade transformers spacy-transformers
    run_ensemble_first_book(CSV_INPUT, OUTPUT_DIR)

/Users/kseniazavyalova/.pyenv/versions/3.11.0/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


[INFO] Загрузка моделей NER...
[ERROR] Ошибка загрузки моделей: [E050] Can't find model 'en_core_web_trf'. It doesn't seem to be a Python package or a valid path to a data directory.
Выполните: pip install --upgrade transformers spacy-transformers


SystemExit: 

In [5]:
%pip install --quiet --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [8]:
import spacy
nlp = spacy.load("en_core_web_trf")

OSError: [E050] Can't find model 'en_core_web_trf'. It doesn't seem to be a Python package or a valid path to a data directory.

In [4]:
import spacy
from transformers import pipeline
import time
import warnings
warnings.filterwarnings('ignore')

DEVICE = -1  # CPU
SPACY_MODEL = "en_core_web_trf"

def safe_pipeline(name, model_id, retries=3, delay=10):
    for i in range(retries):
        try:
            print(f"  ⬇️ Загрузка {name} (попытка {i+1})...")
            return pipeline("ner", model=model_id, aggregation_strategy="simple", device=DEVICE)
        except Exception as e:
            print(f"  ❌ Ошибка: {e}")
            if i < retries - 1:
                print(f"  ⏳ Ожидание {delay}c перед повтором...")
                time.sleep(delay)
    return None

print("[INFO] Инициализация NER...")

# 1. spaCy (локально, не требует интернета после загрузки)
try:
    spacy_ner = spacy.load(SPACY_MODEL)
    print("  ✅ spaCy transformer готов")
except:
    raise SystemExit("❌ Установите модель: !python -m spacy download en_core_web_trf")

# 2. Hugging Face модели (с защитой от обрывов соединения)
bert_pipe = safe_pipeline("BERT", "dslim/bert-base-NER")
roberta_pipe = safe_pipeline("RoBERTa", "Jean-Baptiste/roberta-large-ner-english")

if bert_pipe is None and roberta_pipe is None:
    print("\n⚠️ HF модели недоступны. Будет использован только spaCy + статистический фолбэк.")
else:
    print("\n✅ Модели загружены. Готово к извлечению.")

[INFO] Инициализация NER...


SystemExit: ❌ Установите модель: !python -m spacy download en_core_web_trf

In [6]:
!pip install spacy spacy-transformers

  Using cached spacy_transformers-1.4.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (7.0 kB)
  Using cached thinc-8.3.13-cp311-cp311-macosx_11_0_arm64.whl.metadata (14 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 3.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.5/657.5 kB 3.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 3.1 MB/s  0:00:00 eta 0:00:01
Using cached spacy_transformers-1.4.0-cp311-cp311-macosx_11_0_arm64.whl (166 kB)
  Attempting uninstall: srsly
    Found existing installation: srsly 2.5.1
    Uninstalling srsly-2.5.1:
      Successfully uninstalled srsly-2.5.1
  Attempting uninstall: confection
    Found existing installation: confection 0.1.5━━━━━━━━━━━━━━━━━ 1/6 [confection]
    Uninstalling confection-0.1.5:━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [confection]
      Successfully uninstalled confection-0.1.5━━━━━━━━━━━━━━━ 1/6 [confection]
  At

In [10]:
import spacy
from transformers import pipeline
import pandas as pd
import re
import os
import warnings
from collections import defaultdict
import time
import gc

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

DEVICE = -1  # CPU
CHUNK_SIZE = 1200  # Оптимально для XLM-RoBERTa-base на CPU

print("[INFO] Загрузка моделей...")

# 1. spaCy (статистическая, быстрая)
spacy_ner = spacy.load("en_core_web_sm")
print("  ✅ spaCy (en_core_web_sm)")

# 2. BERT-base NER
try:
    print("  ⬇️ Загрузка BERT-base NER...")
    bert_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=DEVICE)
    print("  ✅ BERT-base NER")
except Exception as e:
    print(f"  ⚠️ BERT-base не загружен: {e}")
    bert_pipe = None

# 3. XLM-RoBERTa-base NER (норм размер, мультязычная, устойчива к контексту)
try:
    print("  ⬇️ Загрузка XLM-RoBERTa-base NER (~550MB)...")
    xlm_pipe = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", aggregation_strategy="simple", device=DEVICE)
    print("  ✅ XLM-RoBERTa-base NER")
except Exception as e:
    print(f"  ⚠️ XLM-RoBERTa не загружен: {e}")
    xlm_pipe = None

gc.collect()

STOP_NAMES = {
    'god', 'christ', 'jesus', 'satan', 'devil', 'hell', 'heaven', 'lord',
    'england', 'london', 'france', 'america', 'europe', 'world', 'earth',
    'time', 'day', 'night', 'year', 'month', 'week', 'monday', 'sunday',
    'mr', 'mrs', 'miss', 'ms', 'dr', 'sir', 'madam', 'lady', 'captain',
    'colonel', 'general', 'reverend', 'father', 'sister', 'brother', 'king',
    'queen', 'prince', 'princess', 'duke', 'duchess', 'earl', 'count'
}

def normalize_name(name: str) -> str:
    name = re.sub(r'[^a-zA-Z\s]', '', name).lower().strip()
    name = re.sub(r'^(mr|mrs|miss|ms|dr|sir|madam|lady|lord|captain|colonel|general|reverend|father|sister)\s+', '', name)
    return ' '.join(name.split())

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    words = text.split()
    chunks, cur, clen = [], [], 0
    for w in words:
        cur.append(w); clen += len(w)+1
        if clen >= chunk_size:
            chunks.append(" ".join(cur)); cur, clen = [], 0
    if cur: chunks.append(" ".join(cur))
    return chunks

# === ЭКСТРАКТОРЫ ===
def extract_spacy_sm(text):
    doc = spacy_ner(text)
    return [{"name": normalize_name(e.text), "conf": 0.75, "method": "spacy_sm"}
            for e in doc.ents if e.label_ == "PERSON" and normalize_name(e.text) not in STOP_NAMES]

def extract_bert_base(text):
    if bert_pipe is None: return []
    out = []
    for chunk in chunk_text(text):
        if not chunk.strip(): continue
        for r in bert_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace(" ##", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "bert_base"})
    return out

def extract_xlm_roberta(text):
    if xlm_pipe is None: return []
    out = []
    for chunk in chunk_text(text, chunk_size=1000):  # Чуть меньше для стабильности
        if not chunk.strip(): continue
        for r in xlm_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace("▁", "")  # XLM-R использует ▁ вместо пробелов
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "xlm_roberta_base"})
    return out

def extract_statistical(text, min_freq=3):
    doc = spacy_ner(text)
    propn_counts = defaultdict(int)
    title_pat = re.compile(r'\b(Mr|Mrs|Miss|Ms|Sir|Lady|Lord|Captain|Colonel|General|Reverend|Father|Sister)\.\?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)')
    
    for t in doc:
        if t.pos_ == "PROPN" and len(t.text) > 2:
            n = normalize_name(t.text)
            if n and n not in STOP_NAMES: propn_counts[n] += 1
            
    res = [{"name": k, "conf": 0.50 + 0.05*min(v,6), "method": "stat_propn"} 
           for k,v in propn_counts.items() if v>=min_freq]
    
    for m in title_pat.finditer(text):
        n = normalize_name(m.group(2))
        if n and n not in STOP_NAMES: 
            res.append({"name": n, "conf": 0.75, "method": "stat_title"})
    return res

# === ЗАПУСК ===
CSV_PATH = "/Users/kseniazavyalova/Desktop/all_books_data.csv"
OUT_DIR = "/Users/kseniazavyalova/Desktop/ner_results"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
text = str(df.iloc[0]['Text'])
book_title = str(df.iloc[0]['Title'])[:30].replace(' ', '_')

print(f"\n📖 Книга: {book_title} | Символов: {len(text):,}")
print("[INFO] Запуск извлечения...")
start_time = time.time()

raw = []
raw.extend(extract_spacy_sm(text)); print("  ✅ spaCy_sm")
raw.extend(extract_bert_base(text)); print("  ✅ BERT-base" if bert_pipe else "  ⏭️ пропущен")
raw.extend(extract_xlm_roberta(text)); print("  ✅ XLM-RoBERTa-base" if xlm_pipe else "  ⏭️ пропущена")
raw.extend(extract_statistical(text)); print("  ✅ Статистика")

elapsed = time.time() - start_time
print(f"⏱️ Время обработки: {elapsed:.1f} сек")

if not raw:
    print("⚠️ Имена не извлечены.")
else:
    df_res = pd.DataFrame(raw)
    
    summary = df_res.groupby('method').agg(
        Extracted_Names=('name', lambda x: ', '.join(sorted(x.unique()))),
        Unique_Count=('name', 'nunique'),
        Avg_Confidence=('conf', 'mean')
    ).round(4).reset_index()
    
    pivot = df_res.pivot_table(index='name', columns='method', values='conf', aggfunc='first').fillna(0)
    
    out_path = os.path.join(OUT_DIR, f"ner_{book_title}.csv")
    pivot_path = os.path.join(OUT_DIR, f"ner_{book_title}_matrix.csv")
    
    summary.to_csv(out_path, index=False)
    pivot.to_csv(pivot_path)
    
    print(f"\n📊 Сводка: {out_path}")
    print(f"🔗 Матрица согласования: {pivot_path}")
    display(summary)
    
    consensus = df_res.groupby('name')['method'].nunique()
    high_conf = consensus[consensus >= 2].index.tolist()
    if high_conf:
        print(f"\n🎯 Имена, найденные ≥2 методами (топ-15):")
        for name in high_conf[:15]:
            methods = df_res[df_res['name']==name]['method'].tolist()
            print(f"  • {name:<25} → {', '.join(methods)}")

[INFO] Загрузка моделей...
  ✅ spaCy (en_core_web_sm)
  ⬇️ Загрузка BERT-base NER...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cpu


  ✅ XLM-RoBERTa-base NER

📖 Книга: A_Feast_for_Crows | Символов: 1,713,576
[INFO] Запуск извлечения...


ValueError: [E088] Text of length 1713576 exceeds maximum of 1000000. The parser and NER models require roughly 1GB of temporary memory per 100,000 characters in the input. This means long texts may cause memory allocation errors. If you're not using the parser or NER, it's probably safe to increase the `nlp.max_length` limit. The limit is in number of characters, so you can check whether your inputs are too long by checking `len(text)`.

In [12]:
import spacy
from transformers import pipeline
import pandas as pd
import re
import os
import warnings
from collections import defaultdict
import time
import gc

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

DEVICE = -1  # CPU
CHUNK_SIZE = 1200  # слов на чанк для трансформеров
SPACY_CHUNK_CHARS = 50000  # символов на чанк для spaCy

print("[INFO] Загрузка моделей...")

# 1. spaCy с увеличенным лимитом длины
spacy_ner = spacy.load("en_core_web_sm")
spacy_ner.max_length = 2_500_000  # Увеличиваем лимит до 2.5 млн символов
print("  ✅ spaCy (en_core_web_sm, max_length=2.5M)")

# 2. BERT-base NER
try:
    print("  ⬇️ Загрузка BERT-base NER...")
    bert_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=DEVICE)
    print("  ✅ BERT-base NER")
except Exception as e:
    print(f"  ⚠️ BERT-base не загружен: {e}")
    bert_pipe = None

# 3. XLM-RoBERTa-base NER
try:
    print("  ⬇️ Загрузка XLM-RoBERTa-base NER (~550MB)...")
    xlm_pipe = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", aggregation_strategy="simple", device=DEVICE)
    print("  ✅ XLM-RoBERTa-base NER")
except Exception as e:
    print(f"  ⚠️ XLM-RoBERTa не загружен: {e}")
    xlm_pipe = None

gc.collect()

STOP_NAMES = {
    'god', 'christ', 'jesus', 'satan', 'devil', 'hell', 'heaven', 'lord',
    'england', 'london', 'france', 'america', 'europe', 'world', 'earth',
    'time', 'day', 'night', 'year', 'month', 'week', 'monday', 'sunday',
    'mr', 'mrs', 'miss', 'ms', 'dr', 'sir', 'madam', 'lady', 'captain',
    'colonel', 'general', 'reverend', 'father', 'sister', 'brother', 'king',
    'queen', 'prince', 'princess', 'duke', 'duchess', 'earl', 'count'
}

def normalize_name(name: str) -> str:
    name = re.sub(r'[^a-zA-Z\s]', '', name).lower().strip()
    name = re.sub(r'^(mr|mrs|miss|ms|dr|sir|madam|lady|lord|captain|colonel|general|reverend|father|sister)\s+', '', name)
    return ' '.join(name.split())

def chunk_text_words(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    """Разбивает текст на чанки по словам (для трансформеров)"""
    words = text.split()
    chunks, cur, clen = [], [], 0
    for w in words:
        cur.append(w); clen += len(w)+1
        if clen >= chunk_size:
            chunks.append(" ".join(cur)); cur, clen = [], 0
    if cur: chunks.append(" ".join(cur))
    return chunks

def chunk_text_chars(text: str, chunk_size: int = SPACY_CHUNK_CHARS) -> list[str]:
    """Разбивает текст на чанки по символам с перекрытием (для spaCy)"""
    chunks = []
    start = 0
    overlap = 500  # перекрытие для сохранения контекста на границах
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Если не конец текста, ищем ближайший пробел/предложение для аккуратной обрезки
        if end < len(text):
            # Ищем ближайший разрыв строки или точку с пробелом
            cut_point = text.rfind('. ', start, end + overlap)
            if cut_point > start + chunk_size // 2:
                end = cut_point + 2
        chunks.append(text[start:end])
        start = end - overlap if end < len(text) else end
    return chunks

# === ЭКСТРАКТОРЫ ===
def extract_spacy_sm(text):
    results = []
    for chunk in chunk_text_chars(text):
        if not chunk.strip(): continue
        doc = spacy_ner(chunk)
        for e in doc.ents:
            if e.label_ == "PERSON":
                norm = normalize_name(e.text)
                if norm not in STOP_NAMES:
                    results.append({"name": norm, "conf": 0.75, "method": "spacy_sm"})
    return results

def extract_bert_base(text):
    if bert_pipe is None: return []
    out = []
    for chunk in chunk_text_words(text):
        if not chunk.strip(): continue
        for r in bert_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace(" ##", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "bert_base"})
    return out

def extract_xlm_roberta(text):
    if xlm_pipe is None: return []
    out = []
    for chunk in chunk_text_words(text, chunk_size=1000):
        if not chunk.strip(): continue
        for r in xlm_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace("▁", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "xlm_roberta_base"})
    return out

def extract_statistical(text, min_freq=3):
    # Для статистики тоже чанкуем, чтобы не перегружать память
    doc = spacy_ner(text[:spacy_ner.max_length])  # берём только до лимита для частот
    propn_counts = defaultdict(int)
    title_pat = re.compile(r'\b(Mr|Mrs|Miss|Ms|Sir|Lady|Lord|Captain|Colonel|General|Reverend|Father|Sister)\.\?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)')
    
    for t in doc:
        if t.pos_ == "PROPN" and len(t.text) > 2:
            n = normalize_name(t.text)
            if n and n not in STOP_NAMES: propn_counts[n] += 1
            
    res = [{"name": k, "conf": 0.50 + 0.05*min(v,6), "method": "stat_propn"} 
           for k,v in propn_counts.items() if v>=min_freq]
    
    for m in title_pat.finditer(text):
        n = normalize_name(m.group(2))
        if n and n not in STOP_NAMES: 
            res.append({"name": n, "conf": 0.75, "method": "stat_title"})
    return res

# === ЗАПУСК ===
CSV_PATH = "/Users/kseniazavyalova/Desktop/all_books_data.csv"
OUT_DIR = "/Users/kseniazavyalova/Desktop/ner_results"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)
text = str(df.iloc[1]['Text'])
book_title = str(df.iloc[2]['Title'])[:30].replace(' ', '_')

print(f"\n📖 Книга: {book_title}")
print(f"📏 Длина текста: {len(text):,} символов")
print("[INFO] Запуск извлечения...")
start_time = time.time()

raw = []
raw.extend(extract_spacy_sm(text)); print("  ✅ spaCy_sm")
raw.extend(extract_bert_base(text)); print("  ✅ BERT-base" if bert_pipe else "  ⏭️ пропущен")
raw.extend(extract_xlm_roberta(text)); print("  ✅ XLM-RoBERTa-base" if xlm_pipe else "  ⏭️ пропущена")
raw.extend(extract_statistical(text)); print("  ✅ Статистика")

elapsed = time.time() - start_time
print(f"⏱️ Время обработки: {elapsed:.1f} сек")

if not raw:
    print("⚠️ Имена не извлечены.")
else:
    df_res = pd.DataFrame(raw)
    
    # Убираем дубликаты (одно имя может встретиться в нескольких чанках)
    df_res = df_res.drop_duplicates(subset=['name', 'method'], keep='first')
    
    summary = df_res.groupby('method').agg(
        Extracted_Names=('name', lambda x: ', '.join(sorted(x.unique()))),
        Unique_Count=('name', 'nunique'),
        Avg_Confidence=('conf', 'mean')
    ).round(4).reset_index()
    
    pivot = df_res.pivot_table(index='name', columns='method', values='conf', aggfunc='first').fillna(0)
    
    out_path = os.path.join(OUT_DIR, f"ner_{book_title}.csv")
    pivot_path = os.path.join(OUT_DIR, f"ner_{book_title}_matrix.csv")
    
    summary.to_csv(out_path, index=False)
    pivot.to_csv(pivot_path)
    
    print(f"\n📊 Сводка: {out_path}")
    print(f"🔗 Матрица согласования: {pivot_path}")
    display(summary)
    
    consensus = df_res.groupby('name')['method'].nunique()
    high_conf = consensus[consensus >= 2].index.tolist()
    if high_conf:
        print(f"\n🎯 Имена, найденные ≥2 методами (топ-15):")
        for name in high_conf[:15]:
            methods = df_res[df_res['name']==name]['method'].tolist()
            print(f"  • {name:<25} → {', '.join(methods)}")

[INFO] Загрузка моделей...
  ✅ spaCy (en_core_web_sm, max_length=2.5M)
  ⬇️ Загрузка BERT-base NER...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f241d9b2-d63f-463d-b4cf-6aff1045e10a)')' thrown while requesting HEAD https://huggingface.co/dslim/bert-base-NER/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


  ✅ BERT-base NER
  ⬇️ Загрузка XLM-RoBERTa-base NER (~550MB)...


Device set to use cpu


  ✅ XLM-RoBERTa-base NER

📖 Книга: BRIDE
📏 Длина текста: 687,792 символов
[INFO] Запуск извлечения...
  ✅ spaCy_sm
  ✅ BERT-base
  ✅ XLM-RoBERTa-base
  ✅ Статистика
⏱️ Время обработки: 192.6 сек

📊 Сводка: /Users/kseniazavyalova/Desktop/ner_results/ner_BRIDE.csv
🔗 Матрица согласования: /Users/kseniazavyalova/Desktop/ner_results/ner_BRIDE_matrix.csv


,method,Extracted_Names,Unique_Count,Avg_Confidence
0,bert_base,", a, aba, ack, ackal, aco, aco palad, ak, aka,...",292,0.7014
1,spacy_sm,"ackal, ackal ergot, ackal ergots, allacath, am...",233,0.7500
2,stat_propn,"ackal, acorn, allacath, amaltar, ambrodel, ari...",253,0.7571
3,xlm_roberta_base,", a, ac, ackal, ackal ergot, ackal ii, ackal i...",396,0.8798



🎯 Имена, найденные ≥2 методами (топ-15):
  •                           → bert_base, xlm_roberta_base
  • a                         → bert_base, xlm_roberta_base
  • ackal                     → spacy_sm, bert_base, xlm_roberta_base, stat_propn
  • ackal ergot               → spacy_sm, xlm_roberta_base
  • acorn                     → xlm_roberta_base, stat_propn
  • ak                        → bert_base, xlm_roberta_base
  • al                        → bert_base, xlm_roberta_base
  • allacath                  → spacy_sm, xlm_roberta_base, stat_propn
  • amal                      → bert_base, xlm_roberta_base
  • amaltar                   → spacy_sm, bert_base, xlm_roberta_base, stat_propn
  • aran                      → bert_base, xlm_roberta_base
  • arise                     → spacy_sm, stat_propn
  • as                        → bert_base, xlm_roberta_base
  • bakal                     → spacy_sm, bert_base, xlm_roberta_base, stat_propn
  • balif                     → xlm_roberta_base

In [13]:
import spacy
from transformers import pipeline
import pandas as pd
import re
import os
import warnings
from collections import defaultdict
import time
import gc
from tqdm import tqdm

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

DEVICE = -1
CHUNK_SIZE = 1200
SPACY_CHUNK_CHARS = 50000

print("[INFO] Загрузка моделей...")

# Модели загружаются ОДИН РАЗ перед циклом
spacy_ner = spacy.load("en_core_web_sm")
spacy_ner.max_length = 2_500_000

try:
    bert_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=DEVICE)
except: bert_pipe = None

try:
    xlm_pipe = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", aggregation_strategy="simple", device=DEVICE)
except: xlm_pipe = None

gc.collect()

STOP_NAMES = {
    'god', 'christ', 'jesus', 'satan', 'devil', 'hell', 'heaven', 'lord',
    'england', 'london', 'france', 'america', 'europe', 'world', 'earth',
    'time', 'day', 'night', 'year', 'month', 'week', 'monday', 'sunday',
    'mr', 'mrs', 'miss', 'ms', 'dr', 'sir', 'madam', 'lady', 'captain',
    'colonel', 'general', 'reverend', 'father', 'sister', 'brother', 'king',
    'queen', 'prince', 'princess', 'duke', 'duchess', 'earl', 'count'
}

def normalize_name(name: str) -> str:
    name = re.sub(r'[^a-zA-Z\s]', '', name).lower().strip()
    name = re.sub(r'^(mr|mrs|miss|ms|dr|sir|madam|lady|lord|captain|colonel|general|reverend|father|sister)\s+', '', name)
    return ' '.join(name.split())

def chunk_text_words(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    words = text.split()
    chunks, cur, clen = [], [], 0
    for w in words:
        cur.append(w); clen += len(w)+1
        if clen >= chunk_size:
            chunks.append(" ".join(cur)); cur, clen = [], 0
    if cur: chunks.append(" ".join(cur))
    return chunks

def chunk_text_chars(text: str, chunk_size: int = SPACY_CHUNK_CHARS) -> list[str]:
    chunks = []
    start = 0
    overlap = 500
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            cut_point = text.rfind('. ', start, end + overlap)
            if cut_point > start + chunk_size // 2:
                end = cut_point + 2
        chunks.append(text[start:end])
        start = end - overlap if end < len(text) else end
    return chunks

def extract_spacy_sm(text):
    results = []
    for chunk in chunk_text_chars(text):
        if not chunk.strip(): continue
        doc = spacy_ner(chunk)
        for e in doc.ents:
            if e.label_ == "PERSON":
                norm = normalize_name(e.text)
                if norm not in STOP_NAMES:
                    results.append({"name": norm, "conf": 0.75, "method": "spacy_sm"})
    return results

def extract_bert_base(text):
    if bert_pipe is None: return []
    out = []
    for chunk in chunk_text_words(text):
        if not chunk.strip(): continue
        for r in bert_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace(" ##", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "bert_base"})
    return out

def extract_xlm_roberta(text):
    if xlm_pipe is None: return []
    out = []
    for chunk in chunk_text_words(text, chunk_size=1000):
        if not chunk.strip(): continue
        for r in xlm_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace("▁", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "xlm_roberta_base"})
    return out

def extract_statistical(text, min_freq=3):
    doc = spacy_ner(text[:spacy_ner.max_length])
    propn_counts = defaultdict(int)
    title_pat = re.compile(r'\b(Mr|Mrs|Miss|Ms|Sir|Lady|Lord|Captain|Colonel|General|Reverend|Father|Sister)\.\?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)')
    for t in doc:
        if t.pos_ == "PROPN" and len(t.text) > 2:
            n = normalize_name(t.text)
            if n and n not in STOP_NAMES: propn_counts[n] += 1
    res = [{"name": k, "conf": 0.50 + 0.05*min(v,6), "method": "stat_propn"} for k,v in propn_counts.items() if v>=min_freq]
    for m in title_pat.finditer(text):
        n = normalize_name(m.group(2))
        if n and n not in STOP_NAMES: 
            res.append({"name": n, "conf": 0.75, "method": "stat_title"})
    return res

def process_single_book(text, book_title, out_dir):
    """Обрабатывает одну книгу и сохраняет результаты"""
    safe_title = re.sub(r'[^\w\-_]', '_', book_title)[:40]
    
    raw = []
    try:
        raw.extend(extract_spacy_sm(text))
        raw.extend(extract_bert_base(text))
        raw.extend(extract_xlm_roberta(text))
        raw.extend(extract_statistical(text))
    except Exception as e:
        print(f"  ❌ Ошибка обработки: {e}")
        return None
    
    if not raw:
        return None
        
    df_res = pd.DataFrame(raw).drop_duplicates(subset=['name', 'method'], keep='first')
    
    summary = df_res.groupby('method').agg(
        Extracted_Names=('name', lambda x: ', '.join(sorted(x.unique()))),
        Unique_Count=('name', 'nunique'),
        Avg_Confidence=('conf', 'mean')
    ).round(4).reset_index()
    
    pivot = df_res.pivot_table(index='name', columns='method', values='conf', aggfunc='first').fillna(0)
    
    # Сохранение файлов для книги
    summary_path = os.path.join(out_dir, f"ner_{safe_title}.csv")
    matrix_path = os.path.join(out_dir, f"ner_{safe_title}_matrix.csv")
    summary.to_csv(summary_path, index=False)
    pivot.to_csv(matrix_path)
    
    return {
        'title': book_title,
        'safe_title': safe_title,
        'total_unique_names': df_res['name'].nunique(),
        'methods_used': df_res['method'].nunique(),
        'high_confidence_count': (df_res.groupby('name')['method'].nunique() >= 2).sum(),
        'summary_path': summary_path,
        'matrix_path': matrix_path
    }

# === ПАКЕТНАЯ ОБРАБОТКА ===
CSV_PATH = "/Users/kseniazavyalova/Desktop/all_books_data.csv"
OUT_DIR = "/Users/kseniazavyalova/Desktop/ner_results_batch"
os.makedirs(OUT_DIR, exist_ok=True)

df_books = pd.read_csv(CSV_PATH)
print(f"📚 Всего книг для обработки: {len(df_books)}")

results_log = []
start_all = time.time()

for idx in tqdm(range(len(df_books)), desc="Обработка книг"):
    try:
        row = df_books.iloc[idx]
        text = str(row.get('Text', ''))
        title = str(row.get('Title', f'Book_{idx}'))
        
        if len(text) < 1000:  # Пропускаем пустые/битые записи
            continue
            
        print(f"\n[{idx+1}/{len(df_books)}] {title[:50]}...")
        book_start = time.time()
        
        result = process_single_book(text, title, OUT_DIR)
        
        if result:
            results_log.append(result)
            elapsed = time.time() - book_start
            print(f"  ✅ Готово за {elapsed:.1f}c | Уникальных имён: {result['total_unique_names']}")
        else:
            print(f"  ⚠️ Пропущено (нет данных или ошибка)")
            
        gc.collect()  # Освобождаем память между книгами
        
    except Exception as e:
        print(f"  ❌ Критическая ошибка: {e}")
        continue

# === ФИНАЛЬНЫЙ ОТЧЁТ ===
total_time = time.time() - start_all
print(f"\n{'='*60}")
print(f"✅ ОБРАБОТКА ЗАВЕРШЕНА")
print(f"📊 Успешно: {len(results_log)} / {len(df_books)}")
print(f"⏱️ Общее время: {total_time/60:.1f} мин")
print(f"📁 Результаты: {OUT_DIR}")

# Сводная таблица по всем книгам
if results_log:
    log_df = pd.DataFrame(results_log)
    log_path = os.path.join(OUT_DIR, "batch_summary.csv")
    log_df.to_csv(log_path, index=False)
    
    print(f"\n📈 Статистика:")
    print(f"  • Всего уникальных имён (сумма): {log_df['total_unique_names'].sum():,}")
    print(f"  • Среднее имён на книгу: {log_df['total_unique_names'].mean():.1f}")
    print(f"  • Книг с ≥2 методами согласия: {(log_df['high_confidence_count'] > 0).sum()}")
    print(f"\n📄 Сводный отчёт: {log_path}")
    display(log_df[['title', 'total_unique_names', 'high_confidence_count']].head(10))

[INFO] Загрузка моделей...


'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: cebf703c-17a0-43fb-a6df-d636d7a76d2c)')' thrown while requesting HEAD https://huggingface.co/dslim/bert-base-NER/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu


📚 Всего книг для обработки: 33


Обработка книг:   0%|          | 0/33 [00:00<?, ?it/s]


[1/33] A Feast for Crows...


...

Обработка книг:  97%|█████████▋| 32/33 [1:43:20<03:43, 223.95s/it]


[33/33] It ends with us...


Обработка книг: 100%|██████████| 33/33 [1:46:01<00:00, 192.77s/it]

  ✅ Готово за 160.7c | Уникальных имён: 262

✅ ОБРАБОТКА ЗАВЕРШЕНА
📊 Успешно: 33 / 33
⏱️ Общее время: 106.0 мин
📁 Результаты: /Users/kseniazavyalova/Desktop/ner_results_batch

📈 Статистика:
  • Всего уникальных имён (сумма): 22,198
  • Среднее имён на книгу: 672.7
  • Книг с ≥2 методами согласия: 33

📄 Сводный отчёт: /Users/kseniazavyalova/Desktop/ner_results_batch/batch_summary.csv


,title,total_unique_names,high_confidence_count
0,A Feast for Crows,5097,1858
1,A Warrior_s Journey,817,249
2,BRIDE,470,162
3,A Street Cat Named Bob,246,74
4,City of Glass,603,190
5,The Thief-Taker's Blade,125,30
6,Fourth Wing,803,232
7,Gideon the Ninth,605,158
8,CRESCENT: CITY HOUSE OF FLAME AND SHADOW,1156,418
9,Ice Station Zebra,396,124


In [15]:
import csv
import requests
import time
import json
from pathlib import Path

# Путь к итоговому CSV-файлу
OUTPUT_CSV = Path("/Users/kseniazavyalova/Desktop/all_books_data2.csv")

HEADERS = {
    "User-Agent": "GenderAdjectivesResearch/1.0 (academic project; contact: your@email.edu)"
}

CORPUS = {
    #  Авторы-женщины (40 книг)
    6053: {"title": "Evelina", "author": "Burney_Fanny", "gender": "female", "genre": "satire", "year": 1778},
    3268: {"title": "The_Mysteries_of_Udolpho", "author": "Radcliffe_Ann", "gender": "female", "genre": "gothic", "year": 1794, "alt_name": True},
    16357: {"title": "Mary_A_Fiction", "author": "Wollstonecraft_Mary", "gender": "female", "genre": "philosophical", "year": 1788},
    171: {"title": "Charlotte_Temple", "author": "Rowson_Susanna", "gender": "female", "genre": "sentimental", "year": 1791},
    161: {"title": "Sense_and_Sensibility", "author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1811, "alt_name": True},
    1342: {"title": "Pride_and_Prejudice", "author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1813, "alt_name": True},
    158: {"title": "Emma", "author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1815, "alt_name": True},
    105: {"title": "Persuasion", "author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1818, "alt_name": True},
    84: {"title": "Frankenstein", "author": "Shelley_Mary", "gender": "female", "genre": "gothic_scifi", "year": 1818, "alt_name": True},
    30863: {"title": "The_Last_Man", "author": "Shelley_Mary", "gender": "female", "genre": "scifi", "year": 1826},
    121: {"title": "Northanger_Abbey", "author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1817, "alt_name": True},
    969: {"title": "The_Tenant_of_Wildfell_Hall", "author": "Bronte_Anne", "gender": "female", "genre": "realism", "year": 1848, "alt_name": True},
    1260: {"title": "Jane_Eyre", "author": "Bronte_Charlotte", "gender": "female", "genre": "gothic", "year": 1847, "alt_name": True},
    841: {"title": "Villette", "author": "Bronte_Charlotte", "gender": "female", "genre": "psychological", "year": 1853, "alt_name": True},
    855: {"title": "Shirley", "author": "Bronte_Charlotte", "gender": "female", "genre": "social", "year": 1849, "alt_name": True},
    768: {"title": "Wuthering_Heights", "author": "Bronte_Emily", "gender": "female", "genre": "gothic", "year": 1847, "alt_name": True},
    1488: {"title": "Agnes_Grey", "author": "Bronte_Anne", "gender": "female", "genre": "realism", "year": 1847},
    145: {"title": "Middlemarch", "author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1871, "alt_name": True},
    550: {"title": "Silas_Marner", "author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1861, "alt_name": True},
    449: {"title": "The_Mill_on_the_Floss", "author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1860},
    1812: {"title": "Adam_Bede", "author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1859},
    4276: {"title": "North_and_South", "author": "Gaskell_Elizabeth", "gender": "female", "genre": "social", "year": 1855, "alt_name": True},
    1493: {"title": "Cranford", "author": "Gaskell_Elizabeth", "gender": "female", "genre": "comedy", "year": 1853, "alt_name": True},
    1492: {"title": "Mary_Barton", "author": "Gaskell_Elizabeth", "gender": "female", "genre": "social", "year": 1848, "alt_name": True},
    160: {"title": "The_Awakening", "author": "Chopin_Kate", "gender": "female", "genre": "psychological", "year": 1899, "alt_name": True},
    30776: {"title": "The_Story_of_an_Hour", "author": "Chopin_Kate", "gender": "female", "genre": "short_story", "year": 1894},
    367: {"title": "The_Country_of_the_Pointed_Firs", "author": "Jewett_Sarah", "gender": "female", "genre": "regionalism", "year": 1896},
    3123: {"title": "A_Country_Doctor", "author": "Jewett_Sarah", "gender": "female", "genre": "regionalism", "year": 1884},
    32: {"title": "Herland", "author": "Gilman_Charlotte", "gender": "female", "genre": "utopia", "year": 1915, "alt_name": True},
    1948: {"title": "The_Yellow_Wallpaper", "author": "Gilman_Charlotte", "gender": "female", "genre": "psychological", "year": 1892},
    541: {"title": "The_Age_of_Innocence", "author": "Wharton_Edith", "gender": "female", "genre": "social", "year": 1920, "alt_name": True},
    28865: {"title": "The_House_of_Mirth", "author": "Wharton_Edith", "gender": "female", "genre": "social", "year": 1905},
    2450: {"title": "Ethan_Frome", "author": "Wharton_Edith", "gender": "female", "genre": "tragedy", "year": 1911, "alt_name": True},
    514: {"title": "Little_Women", "author": "Alcott_Louisa", "gender": "female", "genre": "family", "year": 1868, "alt_name": True},
    2737: {"title": "Little_Men", "author": "Alcott_Louisa", "gender": "female", "genre": "family", "year": 1871},
    515: {"title": "Jos_Boys", "author": "Alcott_Louisa", "gender": "female", "genre": "family", "year": 1886},
    45: {"title": "Anne_of_Green_Gables", "author": "Montgomery_LM", "gender": "female", "genre": "coming_of_age", "year": 1908, "alt_name": True},
    49488: {"title": "Emily_of_New_Moon", "author": "Montgomery_LM", "gender": "female", "genre": "coming_of_age", "year": 1923},
    113: {"title": "The_Secret_Garden", "author": "Burnett_Frances", "gender": "female", "genre": "children", "year": 1911, "alt_name": True},
    16351: {"title": "A_Little_Princess", "author": "Burnett_Frances", "gender": "female", "genre": "children", "year": 1905},

    #  Авторы-мужчины (40 книг)
    6593: {"title": "Tom_Jones", "author": "Fielding_Henry", "gender": "male", "genre": "picaresque", "year": 1749},
    829: {"title": "Gullivers_Travels", "author": "Swift_Jonathan", "gender": "male", "genre": "satire", "year": 1726, "alt_name": True},
    521: {"title": "Robinson_Crusoe", "author": "Defoe_Daniel", "gender": "male", "genre": "adventure", "year": 1719, "alt_name": True},
    1079: {"title": "Tristram_Shandy", "author": "Sterne_Laurence", "gender": "male", "genre": "experimental", "year": 1759},
    4292: {"title": "The_Castle_of_Otranto", "author": "Walpole_Horace", "gender": "male", "genre": "gothic", "year": 1764},
    3974: {"title": "The_Monk", "author": "Lewis_Matthew", "gender": "male", "genre": "gothic", "year": 1796},
    33: {"title": "The_Scarlet_Letter", "author": "Hawthorne_Nathaniel", "gender": "male", "genre": "historical", "year": 1850, "alt_name": True},
    77: {"title": "House_of_Seven_Gables", "author": "Hawthorne_Nathaniel", "gender": "male", "genre": "gothic", "year": 1851, "alt_name": True},
    2489: {"title": "Moby_Dick", "author": "Melville_Herman", "gender": "male", "genre": "adventure", "year": 1851, "alt_name": True},
    11231: {"title": "Bartleby_the_Scrivener", "author": "Melville_Herman", "gender": "male", "genre": "psychological", "year": 1853},
    766: {"title": "David_Copperfield", "author": "Dickens_Charles", "gender": "male", "genre": "realism", "year": 1849, "alt_name": True},
    1400: {"title": "Great_Expectations", "author": "Dickens_Charles", "gender": "male", "genre": "realism", "year": 1861, "alt_name": True},
    730: {"title": "Oliver_Twist", "author": "Dickens_Charles", "gender": "male", "genre": "social", "year": 1838, "alt_name": True},
    1023: {"title": "Bleak_House", "author": "Dickens_Charles", "gender": "male", "genre": "social", "year": 1853, "alt_name": True},
    599: {"title": "Vanity_Fair", "author": "Thackeray_William", "gender": "male", "genre": "satire", "year": 1848, "alt_name": True},
    74: {"title": "Tom_Sawyer", "author": "Twain_Mark", "gender": "male", "genre": "adventure", "year": 1876, "alt_name": True},
    76: {"title": "Huckleberry_Finn", "author": "Twain_Mark", "gender": "male", "genre": "adventure", "year": 1884, "alt_name": True},
    1837: {"title": "Prince_and_the_Pauper", "author": "Twain_Mark", "gender": "male", "genre": "historical", "year": 1881, "alt_name": True},
    110: {"title": "Tess_of_the_dUrbervilles", "author": "Hardy_Thomas", "gender": "male", "genre": "naturalism", "year": 1891, "alt_name": True},
    27: {"title": "Far_from_the_Madding_Crowd", "author": "Hardy_Thomas", "gender": "male", "genre": "realism", "year": 1874, "alt_name": True},
    101: {"title": "Jude_the_Obscure", "author": "Hardy_Thomas", "gender": "male", "genre": "tragedy", "year": 1895, "alt_name": True},
    174: {"title": "Picture_of_Dorian_Gray", "author": "Wilde_Oscar", "gender": "male", "genre": "gothic", "year": 1890, "alt_name": True},
    844: {"title": "Importance_of_Being_Earnest", "author": "Wilde_Oscar", "gender": "male", "genre": "comedy", "year": 1895, "alt_name": True},
    345: {"title": "Dracula", "author": "Stoker_Bram", "gender": "male", "genre": "gothic", "year": 1897, "alt_name": True},
    36: {"title": "War_of_the_Worlds", "author": "Wells_HG", "gender": "male", "genre": "scifi", "year": 1897, "alt_name": True},
    5230: {"title": "The_Invisible_Man", "author": "Wells_HG", "gender": "male", "genre": "scifi", "year": 1897},
    35: {"title": "The_Time_Machine", "author": "Wells_HG", "gender": "male", "genre": "scifi", "year": 1895, "alt_name": True},
    219: {"title": "Heart_of_Darkness", "author": "Conrad_Joseph", "gender": "male", "genre": "psychological", "year": 1899, "alt_name": True},
    202: {"title": "Lord_Jim", "author": "Conrad_Joseph", "gender": "male", "genre": "adventure", "year": 1900, "alt_name": True},
    974: {"title": "The_Secret_Agent", "author": "Conrad_Joseph", "gender": "male", "genre": "political", "year": 1907, "alt_name": True},
    1008: {"title": "Portrait_of_a_Lady", "author": "James_Henry", "gender": "male", "genre": "psychological", "year": 1881, "alt_name": True},
    209: {"title": "Turn_of_the_Screw", "author": "James_Henry", "gender": "male", "genre": "gothic", "year": 1898, "alt_name": True},
    208: {"title": "Daisy_Miller", "author": "James_Henry", "gender": "male", "genre": "psychological", "year": 1878},
    1661: {"title": "Adventures_of_Sherlock_Holmes", "author": "Doyle_Arthur", "gender": "male", "genre": "detective", "year": 1892, "alt_name": True},
    2852: {"title": "Hound_of_the_Baskervilles", "author": "Doyle_Arthur", "gender": "male", "genre": "detective", "year": 1902, "alt_name": True},
    233: {"title": "Sister_Carrie", "author": "Dreiser_Theodore", "gender": "male", "genre": "naturalism", "year": 1900, "alt_name": True},
    2231: {"title": "An_American_Tragedy", "author": "Dreiser_Theodore", "gender": "male", "genre": "naturalism", "year": 1925},
    805: {"title": "This_Side_of_Paradise", "author": "Fitzgerald_FScott", "gender": "male", "genre": "modernist", "year": 1920},
    9830: {"title": "The_Beautiful_and_Damned", "author": "Fitzgerald_FScott", "gender": "male", "genre": "modernist", "year": 1922, "alt_name": True},
}


def clean_gutenberg_text(text):
    """Удаляет заголовки и футеры Project Gutenberg"""
    if "*** START OF" in text:
        text = text.split("*** START OF", 1)[1]
    if "*** END OF" in text:
        text = text.split("*** END OF", 1)[0]

    lines = []
    for line in text.split('\n'):
        lower = line.lower()
        if any(k in lower for k in [
            'project gutenberg', 'ebook', 'copyright',
            'license', 'public domain', 'smashwords'
        ]):
            continue
        lines.append(line)

    return '\n'.join(lines).strip()


def parse_author(author_str):
    """Преобразует формат 'Lastname_Firstname' в 'Firstname Lastname'"""
    if not author_str:
        return "Нет автора"
    parts = author_str.split("_")
    if len(parts) >= 2:
        return f"{parts[1]} {' '.join(parts[2:])}".strip()
    return author_str.replace("_", " ")


def try_download(book_id, use_alt_name=False):
    """Пробует скачать книгу в нескольких форматах"""
    filenames = [
        f"{book_id}-0.txt",
        f"{book_id}.txt",
        f"{book_id}.txt.utf-8",
        f"{book_id}-h.htm",
    ]
    # Меняем порядок поиска в зависимости от alt_name
    if use_alt_name:
        filenames = [filenames[0], filenames[1], filenames[2], filenames[3]]
    else:
        filenames = [filenames[1], filenames[0], filenames[2], filenames[3]]

    base_url = f"https://www.gutenberg.org/files/{book_id}"
    cache_url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"

    for filename in filenames:
        url = f"{base_url}/{filename}"
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                text = _decode_text(resp.content)
                if text and len(text) > 1000:
                    return True, clean_gutenberg_text(text), url
        except requests.RequestException:
            continue

    # Пробуем кэш
    try:
        resp = requests.get(cache_url, headers=HEADERS, timeout=30)
        if resp.status_code == 200:
            text = _decode_text(resp.content)
            if text and len(text) > 1000:
                return True, clean_gutenberg_text(text), cache_url
    except requests.RequestException:
        pass

    return False, None, None


def _decode_text(content):
    """Декодирует байты в текст, пробуя разные кодировки"""
    for encoding in ['utf-8', 'iso-8859-1', 'cp1252', 'latin-1']:
        try:
            return content.decode(encoding, errors='strict')
        except UnicodeDecodeError:
            continue
    return content.decode('utf-8', errors='ignore')


def main():
    results = []
    processed = 0
    failed = 0

    print(f"Начинаю парсинг {len(CORPUS)} книг в {OUTPUT_CSV}...\n")

    # Открываем CSV для записи
    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=["Title", "Authors", "Genre", "Text"])
        writer.writeheader()

        for i, (book_id, meta) in enumerate(CORPUS.items(), 1):
            title_key = meta['title']
            year = meta['year']
            use_alt = meta.get('alt_name', False)

            print(f"[{i}/{len(CORPUS)}] {title_key.replace('_', ' ')} ({year})", end=" ... ")

            # Пробуем скачать
            success, text, used_url = try_download(book_id, use_alt_name=use_alt)

            # Если не получилось, пробуем с обратным флагом alt_name
            if not success:
                success, text, used_url = try_download(book_id, use_alt_name=not use_alt)

            if success and text:
                # Формируем поля для CSV
                title = title_key.replace("_", " ")
                authors_str = parse_author(meta.get("author", ""))
                genre_str = meta.get("genre", "Без жанра")
                clean_text = text.strip()

                # Записываем строку в CSV
                writer.writerow({
                    "Title": title,
                    "Authors": authors_str,
                    "Genre": genre_str,
                    "Text": clean_text
                })

                print(f"[OK] {len(text):,} символов")
                results.append({
                    'id': book_id,
                    'title': title_key,
                    'status': 'success',
                    'chars': len(text),
                    'url': used_url
                })
                processed += 1
            else:
                print("[ERROR]")
                results.append({'id': book_id, 'title': title_key, 'status': 'failed'})
                failed += 1

            time.sleep(2)  # Пауза между запросами

    # Сохраняем отчёт о загрузке рядом с CSV
    report_path = OUTPUT_CSV.parent / "download_report.json"
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n{'='*60}")
    print(f"✅ Готово: {processed} книг записано в CSV")
    print(f"❌ Ошибки: {failed}")
    print(f"📄 Результат: {OUTPUT_CSV}")
    print(f"📋 Отчёт: {report_path}")

    if failed > 0:
        print(f"\n⚠️ Не удалось скачать:")
        for r in results:
            if r['status'] == 'failed':
                print(f"   • {r['title']} (ID: {r['id']})")


if __name__ == "__main__":
    main()

Начинаю парсинг 79 книг в /Users/kseniazavyalova/Desktop/all_books_data2.csv...

[1/79] Evelina (1778) ... [OK] 887,681 символов
[2/79] The Mysteries of Udolpho (1794) ... [OK] 1,710,352 символов
[3/79] Mary A Fiction (1788) ... [OK] 148,866 символов
[4/79] Charlotte Temple (1791) ... [OK] 214,172 символов
[5/79] Sense and Sensibility (1811) ... [OK] 683,346 символов
[6/79] Pride and Prejudice (1813) ... [OK] 727,362 символов
[7/79] Emma (1815) ... [OK] 880,423 символов
[8/79] Persuasion (1818) ... [OK] 473,116 символов
[9/79] Frankenstein (1818) ... [OK] 419,336 символов
[10/79] The Last Man (1826) ... [OK] 765,806 символов
[11/79] Northanger Abbey (1817) ... [OK] 441,994 символов
[12/79] The Tenant of Wildfell Hall (1848) ... [OK] 953,587 символов
[13/79] Jane Eyre (1847) ... [OK] 1,022,061 символов
[14/79] Villette (1853) ... [OK] 251,841 символов
[15/79] Shirley (1849) ... [OK] 21,238 символов
[16/79] Wuthering Heights (1847) ... [OK] 645,605 символов
[17/79] Agnes Grey (1847) ... 

In [ ]:
import spacy
from transformers import pipeline
import pandas as pd
import re
import os
import warnings
from collections import defaultdict
import time
import gc
from tqdm import tqdm

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

DEVICE = -1
CHUNK_SIZE = 1200
SPACY_CHUNK_CHARS = 50000

print("[INFO] Загрузка моделей...")

# Модели загружаются ОДИН РАЗ перед циклом
spacy_ner = spacy.load("en_core_web_sm")
spacy_ner.max_length = 2_500_000

try:
    bert_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=DEVICE)
except: bert_pipe = None

try:
    xlm_pipe = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", aggregation_strategy="simple", device=DEVICE)
except: xlm_pipe = None

gc.collect()

STOP_NAMES = {
    'god', 'christ', 'jesus', 'satan', 'devil', 'hell', 'heaven', 'lord',
    'england', 'london', 'france', 'america', 'europe', 'world', 'earth',
    'time', 'day', 'night', 'year', 'month', 'week', 'monday', 'sunday',
    'mr', 'mrs', 'miss', 'ms', 'dr', 'sir', 'madam', 'lady', 'captain',
    'colonel', 'general', 'reverend', 'father', 'sister', 'brother', 'king',
    'queen', 'prince', 'princess', 'duke', 'duchess', 'earl', 'count'
}

def normalize_name(name: str) -> str:
    name = re.sub(r'[^a-zA-Z\s]', '', name).lower().strip()
    name = re.sub(r'^(mr|mrs|miss|ms|dr|sir|madam|lady|lord|captain|colonel|general|reverend|father|sister)\s+', '', name)
    return ' '.join(name.split())

def chunk_text_words(text: str, chunk_size: int = CHUNK_SIZE) -> list[str]:
    words = text.split()
    chunks, cur, clen = [], [], 0
    for w in words:
        cur.append(w); clen += len(w)+1
        if clen >= chunk_size:
            chunks.append(" ".join(cur)); cur, clen = [], 0
    if cur: chunks.append(" ".join(cur))
    return chunks

def chunk_text_chars(text: str, chunk_size: int = SPACY_CHUNK_CHARS) -> list[str]:
    chunks = []
    start = 0
    overlap = 500
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            cut_point = text.rfind('. ', start, end + overlap)
            if cut_point > start + chunk_size // 2:
                end = cut_point + 2
        chunks.append(text[start:end])
        start = end - overlap if end < len(text) else end
    return chunks

def extract_spacy_sm(text):
    results = []
    for chunk in chunk_text_chars(text):
        if not chunk.strip(): continue
        doc = spacy_ner(chunk)
        for e in doc.ents:
            if e.label_ == "PERSON":
                norm = normalize_name(e.text)
                if norm not in STOP_NAMES:
                    results.append({"name": norm, "conf": 0.75, "method": "spacy_sm"})
    return results

def extract_bert_base(text):
    if bert_pipe is None: return []
    out = []
    for chunk in chunk_text_words(text):
        if not chunk.strip(): continue
        for r in bert_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace(" ##", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "bert_base"})
    return out

def extract_xlm_roberta(text):
    if xlm_pipe is None: return []
    out = []
    for chunk in chunk_text_words(text, chunk_size=1000):
        if not chunk.strip(): continue
        for r in xlm_pipe(chunk):
            if r.get("entity_group") == "PER":
                word = r["word"].replace("▁", "")
                norm = normalize_name(word)
                if norm not in STOP_NAMES:
                    out.append({"name": norm, "conf": float(r["score"]), "method": "xlm_roberta_base"})
    return out

def extract_statistical(text, min_freq=3):
    doc = spacy_ner(text[:spacy_ner.max_length])
    propn_counts = defaultdict(int)
    title_pat = re.compile(r'\b(Mr|Mrs|Miss|Ms|Sir|Lady|Lord|Captain|Colonel|General|Reverend|Father|Sister)\.\?\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)')
    for t in doc:
        if t.pos_ == "PROPN" and len(t.text) > 2:
            n = normalize_name(t.text)
            if n and n not in STOP_NAMES: propn_counts[n] += 1
    res = [{"name": k, "conf": 0.50 + 0.05*min(v,6), "method": "stat_propn"} for k,v in propn_counts.items() if v>=min_freq]
    for m in title_pat.finditer(text):
        n = normalize_name(m.group(2))
        if n and n not in STOP_NAMES: 
            res.append({"name": n, "conf": 0.75, "method": "stat_title"})
    return res

def process_single_book(text, book_title, out_dir):
    """Обрабатывает одну книгу и сохраняет результаты"""
    safe_title = re.sub(r'[^\w\-_]', '_', book_title)[:40]
    
    raw = []
    try:
        raw.extend(extract_spacy_sm(text))
        raw.extend(extract_bert_base(text))
        raw.extend(extract_xlm_roberta(text))
        raw.extend(extract_statistical(text))
    except Exception as e:
        print(f"  ❌ Ошибка обработки: {e}")
        return None
    
    if not raw:
        return None
        
    df_res = pd.DataFrame(raw).drop_duplicates(subset=['name', 'method'], keep='first')
    
    summary = df_res.groupby('method').agg(
        Extracted_Names=('name', lambda x: ', '.join(sorted(x.unique()))),
        Unique_Count=('name', 'nunique'),
        Avg_Confidence=('conf', 'mean')
    ).round(4).reset_index()
    
    pivot = df_res.pivot_table(index='name', columns='method', values='conf', aggfunc='first').fillna(0)
    
    # Сохранение файлов для книги
    summary_path = os.path.join(out_dir, f"ner_{safe_title}.csv")
    matrix_path = os.path.join(out_dir, f"ner_{safe_title}_matrix.csv")
    summary.to_csv(summary_path, index=False)
    pivot.to_csv(matrix_path)
    
    return {
        'title': book_title,
        'safe_title': safe_title,
        'total_unique_names': df_res['name'].nunique(),
        'methods_used': df_res['method'].nunique(),
        'high_confidence_count': (df_res.groupby('name')['method'].nunique() >= 2).sum(),
        'summary_path': summary_path,
        'matrix_path': matrix_path
    }

CSV_PATH = "/Users/kseniazavyalova/Desktop/all_books_data2.csv"
OUT_DIR = "/Users/kseniazavyalova/Desktop/ner_results_batch2"
os.makedirs(OUT_DIR, exist_ok=True)

df_books = pd.read_csv(CSV_PATH)
print(f"📚 Всего книг для обработки: {len(df_books)}")

results_log = []
start_all = time.time()

for idx in tqdm(range(len(df_books)), desc="Обработка книг"):
    try:
        row = df_books.iloc[idx]
        
        # === ПРОВЕРКА НАЛИЧИЯ ТЕКСТА ===
        text_raw = row.get('Text')
        
        # Пропускаем, если текст отсутствует, NaN, не строка или пустой
        if pd.isna(text_raw) or not isinstance(text_raw, str):
            print(f"[{idx+1}/{len(df_books)}] ⚠️ Пропущено: нет текста в поле 'Text'")
            continue
            
        text = text_raw.strip()
        
        if not text or len(text) < 1000:
            print(f"[{idx+1}/{len(df_books)}] ⚠️ Пропущено: текст пустой или слишком короткий ({len(text)} симв.)")
            continue
        # === КОНЕЦ ПРОВЕРКИ ===
        
        title = str(row.get('Title', f'Book_{idx}'))
        
        print(f"\n[{idx+1}/{len(df_books)}] {title[:50]}...")
        book_start = time.time()
        
        result = process_single_book(text, title, OUT_DIR)
        
        if result:
            results_log.append(result)
            elapsed = time.time() - book_start
            print(f"  ✅ Готово за {elapsed:.1f}c | Уникальных имён: {result['total_unique_names']}")
        else:
            print(f"  ⚠️ Пропущено (нет данных или ошибка)")
            
        gc.collect()
        
    except Exception as e:
        print(f"  ❌ Критическая ошибка: {e}")
        continue

# === ФИНАЛЬНЫЙ ОТЧЁТ ===
total_time = time.time() - start_all
print(f"\n{'='*60}")
print(f"✅ ОБРАБОТКА ЗАВЕРШЕНА")
print(f"📊 Успешно: {len(results_log)} / {len(df_books)}")
print(f"⏱️ Общее время: {total_time/60:.1f} мин")
print(f"📁 Результаты: {OUT_DIR}")

# Сводная таблица по всем книгам
if results_log:
    log_df = pd.DataFrame(results_log)
    log_path = os.path.join(OUT_DIR, "batch_summary.csv")
    log_df.to_csv(log_path, index=False)
    
    print(f"\n📈 Статистика:")
    print(f"  • Всего уникальных имён (сумма): {log_df['total_unique_names'].sum():,}")
    print(f"  • Среднее имён на книгу: {log_df['total_unique_names'].mean():.1f}")
    print(f"  • Книг с ≥2 методами согласия: {(log_df['high_confidence_count'] > 0).sum()}")
    print(f"\n📄 Сводный отчёт: {log_path}")
    display(log_df[['title', 'total_unique_names', 'high_confidence_count']].head(10))

[INFO] Загрузка моделей...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 19242ca9-e5e6-420c-a811-bd55498a2ae9)')' thrown while requesting HEAD https://huggingface.co/dslim/bert-base-NER/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu


📚 Всего книг для обработки: 75


Обработка книг:   0%|          | 0/75 [00:00<?, ?it/s]


[1/75] Evelina...
  ✅ Готово за 344.7c | Уникальных имён: 597


...

Обработка книг:  99%|█████████▊| 74/75 [4:33:36<02:53, 173.19s/it]

  ✅ Готово за 194.6c | Уникальных имён: 1103

[75/75] The Beautiful and Damned...


Обработка книг: 100%|██████████| 75/75 [4:38:29<00:00, 222.79s/it]

  ✅ Готово за 292.8c | Уникальных имён: 952

✅ ОБРАБОТКА ЗАВЕРШЕНА
📊 Успешно: 75 / 75
⏱️ Общее время: 278.5 мин
📁 Результаты: /Users/kseniazavyalova/Desktop/ner_results_batch2

📈 Статистика:
  • Всего уникальных имён (сумма): 60,388
  • Среднее имён на книгу: 805.2
  • Книг с ≥2 методами согласия: 75

📄 Сводный отчёт: /Users/kseniazavyalova/Desktop/ner_results_batch2/batch_summary.csv


,title,total_unique_names,high_confidence_count
0,Evelina,597,180
1,The Mysteries of Udolpho,815,239
2,Mary A Fiction,90,23
3,Charlotte Temple,204,62
4,Sense and Sensibility,358,106
5,Pride and Prejudice,407,163
6,Emma,492,152
7,Persuasion,358,110
8,Frankenstein,255,95
9,The Last Man,654,158


In [4]:
import pandas as pd
df = pd.read_parquet('/Users/kseniazavyalova/Downloads/test_data.parquet')
df

,request
0,"[{'role': 'system', 'text': 'You are a classif..."
1,"[{'role': 'system', 'text': 'You are a classif..."
2,"[{'role': 'system', 'text': 'You are a classif..."
3,"[{'role': 'system', 'text': 'You are a classif..."
4,"[{'role': 'system', 'text': 'You are a classif..."
5,"[{'role': 'system', 'text': 'You are a classif..."
6,"[{'role': 'system', 'text': 'You are a classif..."
7,"[{'role': 'system', 'text': 'You are a classif..."
8,"[{'role': 'system', 'text': 'You are a classif..."
9,"[{'role': 'system', 'text': 'You are a classif..."


In [8]:
import pandas as pd
import json

# === Пути ===
INPUT_CSV = "/Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_matrix.csv"
OUTPUT_JSONL = "/Users/kseniazavyalova/Desktop/ner_results_batch/name_classification_prompts.jsonl"

# === Системный промпт ===
SYSTEM_PROMPT = (
    "You are a name classifier for fantasy fiction texts. "
    "All input tokens are already in lowercase — do NOT use "
    "capitalization as a signal.\n\n"
    "For each input token, respond with exactly one digit:\n"
    "  1 — female given name\n"
    "  2 — male given name\n"
    "  3 — not a name (common noun, place, surname, object, "
    "species, abbreviation, etc.)\n\n"
    "Rules:\n"
    "1. Output ONLY the digit (1, 2, or 3).\n"
    "2. No explanations, no punctuation, no extra text.\n"
    "3. Names may be fictional/fantasy names that do not exist "
    "in the real world. Classify them based on whether they "
    "function as a personal given name — not a place, object, "
    "title, or species.\n"
    "4. Ignore capitalization entirely — all inputs are lowercase. "
    "Rely on the lexical and morphological nature of the token, "
    "not its casing.\n"
    "5. If ambiguous, choose the most likely category."
)


def build_request(name: str) -> dict:
    messages = [
        {"role": "system", "text": SYSTEM_PROMPT},
        {"role": "user",   "text": name},
    ]
    return {"request": messages}


def main():
    df = pd.read_csv(INPUT_CSV)

    if "name" not in df.columns:
        raise KeyError(f"В CSV нет колонки 'name'. Найдены: {df.columns.tolist()}")

    records_written = 0

    with open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:
        for raw_name in df["name"]:
            if pd.isna(raw_name):
                continue
            # На всякий случай приводим к нижнему регистру здесь тоже,
            # чтобы промпт не расходился с реальными данными
            name = str(raw_name).strip().lower()
            if not name:
                continue

            request_obj = build_request(name)
            fout.write(json.dumps(request_obj, ensure_ascii=False) + "\n")
            records_written += 1

    print(f"Готово. Записано строк: {records_written}")
    print(f"Результат: {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Готово. Записано строк: 816
Результат: /Users/kseniazavyalova/Desktop/ner_results_batch/name_classification_prompts.jsonl


In [9]:
import pandas as pd
import json
from pathlib import Path

# === Папка и паттерн имён файлов ===
# Обрабатываем все CSV, которые заканчиваются на "_matrix.csv"
DATA_DIR = Path("/Users/kseniazavyalova/Desktop/ner_results_batch")
INPUT_PATTERN = "*_matrix.csv"

# === Системный промпт ===
SYSTEM_PROMPT = (
    "You are a name classifier for fantasy fiction texts. "
    "All input tokens are already in lowercase — do NOT use "
    "capitalization as a signal.\n\n"
    "For each input token, respond with exactly one digit:\n"
    "  1 — female given name\n"
    "  2 — male given name\n"
    "  3 — not a name (common noun, place, surname, object, "
    "species, abbreviation, etc.)\n\n"
    "Rules:\n"
    "1. Output ONLY the digit (1, 2, or 3).\n"
    "2. No explanations, no punctuation, no extra text.\n"
    "3. Names may be fictional/fantasy names that do not exist "
    "in the real world. Classify them based on whether they "
    "function as a personal given name — not a place, object, "
    "title, or species.\n"
    "4. Ignore capitalization entirely — all inputs are lowercase. "
    "Rely on the lexical and morphological nature of the token, "
    "not its casing.\n"
    "5. If ambiguous, choose the most likely category."
)


def build_request(name: str) -> dict:
    messages = [
        {"role": "system", "text": SYSTEM_PROMPT},
        {"role": "user",   "text": name},
    ]
    return {"request": messages}


def extract_book_name(input_path: Path) -> str:
    """
    Извлекает имя книги из имени файла.
    ner_A_Warrior_s_Journey_matrix.csv  ->  A_Warrior_s_Journey
    """
    # Убираем префикс "ner_" и суффикс "_matrix.csv"
    stem = input_path.stem                  # "ner_A_Warrior_s_Journey_matrix"
    if stem.startswith("ner_"):
        stem = stem[len("ner_"):]           # "A_Warrior_s_Journey_matrix"
    if stem.endswith("_matrix"):
        stem = stem[: -len("_matrix")]      # "A_Warrior_s_Journey"
    return stem


def process_file(input_path: Path, output_path: Path) -> int:
    """
    Обрабатывает один CSV и пишет соответствующий JSONL.
    Возвращает количество записанных строк.
    """
    df = pd.read_csv(input_path)

    if "name" not in df.columns:
        raise KeyError(
            f"В файле {input_path.name} нет колонки 'name'. "
            f"Найдены: {df.columns.tolist()}"
        )

    records_written = 0

    with open(output_path, "w", encoding="utf-8") as fout:
        for raw_name in df["name"]:
            if pd.isna(raw_name):
                continue
            name = str(raw_name).strip().lower()
            if not name:
                continue

            request_obj = build_request(name)
            fout.write(json.dumps(request_obj, ensure_ascii=False) + "\n")
            records_written += 1

    return records_written


def main():
    input_files = sorted(DATA_DIR.glob(INPUT_PATTERN))

    if not input_files:
        print(f"В папке {DATA_DIR} не найдено файлов по паттерну '{INPUT_PATTERN}'")
        return

    print(f"Найдено файлов для обработки: {len(input_files)}\n")

    total_records = 0
    processed = 0
    failed = 0

    for input_path in input_files:
        book_name = extract_book_name(input_path)
        output_path = DATA_DIR / f"name_classification_prompts_{book_name}.jsonl"

        try:
            count = process_file(input_path, output_path)
            total_records += count
            processed += 1
            print(f"[OK] {input_path.name}")
            print(f"     -> {output_path.name}  ({count} строк)")
        except Exception as e:
            failed += 1
            print(f"[FAIL] {input_path.name}: {e}")

    print("\n=== Итог ===")
    print(f"Обработано успешно: {processed}")
    print(f"Пропущено с ошибкой: {failed}")
    print(f"Всего записей во всех JSONL: {total_records}")


if __name__ == "__main__":
    main()

Найдено файлов для обработки: 33

[OK] ner_A_Feast_for_Crows_matrix.csv
     -> name_classification_prompts_A_Feast_for_Crows.jsonl  (5095 строк)
[OK] ner_A_Street_Cat_Named_Bob_matrix.csv
     -> name_classification_prompts_A_Street_Cat_Named_Bob.jsonl  (245 строк)
[OK] ner_A_Warrior_s_Journey_matrix.csv
     -> name_classification_prompts_A_Warrior_s_Journey.jsonl  (816 строк)
[OK] ner_BRIDE_matrix.csv
     -> name_classification_prompts_BRIDE.jsonl  (469 строк)
[OK] ner_CHAPTER_ONE_matrix.csv
     -> name_classification_prompts_CHAPTER_ONE.jsonl  (661 строк)
[OK] ner_CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW_matrix.csv
     -> name_classification_prompts_CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW.jsonl  (1155 строк)
[OK] ner_City_of_Glass_matrix.csv
     -> name_classification_prompts_City_of_Glass.jsonl  (602 строк)
[OK] ner_Fourth_Wing_matrix.csv
     -> name_classification_prompts_Fourth_Wing.jsonl  (801 строк)
[OK] ner_Gideon_the_Ninth_matrix.csv
     -> name_classification_prompts

In [10]:
import pandas as pd
import json
from pathlib import Path

# === Папка и паттерн имён файлов ===
DATA_DIR = Path("/Users/kseniazavyalova/Desktop/ner_results_batch")
INPUT_PATTERN = "*_matrix.csv"
OUTPUT_JSONL = DATA_DIR / "name_classification_prompts_all_unique.jsonl"

# === Системный промпт ===
SYSTEM_PROMPT = (
    "You are a name classifier for fantasy fiction texts. "
    "All input tokens are already in lowercase — do NOT use "
    "capitalization as a signal.\n\n"
    "For each input token, respond with exactly one digit:\n"
    "  1 — female given name\n"
    "  2 — male given name\n"
    "  3 — not a name (common noun, place, surname, object, "
    "species, abbreviation, etc.)\n\n"
    "Rules:\n"
    "1. Output ONLY the digit (1, 2, or 3).\n"
    "2. No explanations, no punctuation, no extra text.\n"
    "3. Names may be fictional/fantasy names that do not exist "
    "in the real world. Classify them based on whether they "
    "function as a personal given name — not a place, object, "
    "title, or species.\n"
    "4. Ignore capitalization entirely — all inputs are lowercase. "
    "Rely on the lexical and morphological nature of the token, "
    "not its casing.\n"
    "5. If ambiguous, choose the most likely category."
)


def build_request(name: str) -> dict:
    messages = [
        {"role": "system", "text": SYSTEM_PROMPT},
        {"role": "user",   "text": name},
    ]
    return {"request": messages}


def main():
    input_files = sorted(DATA_DIR.glob(INPUT_PATTERN))

    if not input_files:
        print(f"В папке {DATA_DIR} не найдено файлов по паттерну '{INPUT_PATTERN}'")
        return

    print(f"Найдено файлов для обработки: {len(input_files)}\n")

    # Собираем все уникальные имена
    unique_names = set()
    processed = 0
    failed = 0

    for input_path in input_files:
        try:
            df = pd.read_csv(input_path)

            if "name" not in df.columns:
                raise KeyError(
                    f"В файле {input_path.name} нет колонки 'name'. "
                    f"Найдены: {df.columns.tolist()}"
                )

            for raw_name in df["name"]:
                if pd.isna(raw_name):
                    continue
                name = str(raw_name).strip().lower()
                if name:
                    unique_names.add(name)

            processed += 1
            print(f"[OK] {input_path.name}")

        except Exception as e:
            failed += 1
            print(f"[FAIL] {input_path.name}: {e}")

    print(f"\nСобрано уникальных имен: {len(unique_names)}")

    # Записываем один общий JSONL
    records_written = 0

    with open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:
        for name in sorted(unique_names):  # сортировка для предсказуемого порядка
            request_obj = build_request(name)
            fout.write(json.dumps(request_obj, ensure_ascii=False) + "\n")
            records_written += 1

    print(f"\n=== Итог ===")
    print(f"Обработано файлов: {processed}")
    print(f"Пропущено с ошибкой: {failed}")
    print(f"Уникальных имен: {len(unique_names)}")
    print(f"Записей в JSONL: {records_written}")
    print(f"Результат: {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Найдено файлов для обработки: 33

[OK] ner_A_Feast_for_Crows_matrix.csv
[OK] ner_A_Street_Cat_Named_Bob_matrix.csv
[OK] ner_A_Warrior_s_Journey_matrix.csv
[OK] ner_BRIDE_matrix.csv
[OK] ner_CHAPTER_ONE_matrix.csv
[OK] ner_CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW_matrix.csv
[OK] ner_City_of_Glass_matrix.csv
[OK] ner_Fourth_Wing_matrix.csv
[OK] ner_Gideon_the_Ninth_matrix.csv
[OK] ner_Girl_Online_matrix.csv
[OK] ner_Harry_Potter_and_the_Sorcerer_s_Stone_matrix.csv
[OK] ner_Ice_Station_Zebra_matrix.csv
[OK] ner_It_ends_with_us_matrix.csv
[OK] ner_Juniper___Thorn_matrix.csv
[OK] ner_Mortal_Engines_matrix.csv
[OK] ner_Realms_of_the_Dragons_vol_1_matrix.csv
[OK] ner_Roots_of_Chaos_3__Among_the_Burning_Flow_matrix.csv
[OK] ner_Royal_Fashion_2__All_the_Gossip_From_Par_matrix.csv
[OK] ner_Sands_of_the_Soul_matrix.csv
[OK] ner_Tamora_Pierce_-_The_Will_of_the_matrix.csv
[OK] ner_The_Hobbit_matrix.csv
[OK] ner_The_Mage_In_The_Iron_Mask_matrix.csv
[OK] ner_The_Magician_s_Nephew_matrix.csv
[OK] ner_

In [11]:
import pandas as pd
df = pd.read_parquet('/Users/kseniazavyalova/Downloads/Batch inference fbii29i5t83mpq464gt8 result.parquet')
df

,request,response,error
0,"[{'role': 'system', 'text': 'You are a name cl...",3,None
1,"[{'role': 'system', 'text': 'You are a name cl...",3,None
2,"[{'role': 'system', 'text': 'You are a name cl...",3,None
3,"[{'role': 'system', 'text': 'You are a name cl...",3,None
4,"[{'role': 'system', 'text': 'You are a name cl...",3,None
...,...,...,...
811,"[{'role': 'system', 'text': 'You are a name cl...",3,None
812,"[{'role': 'system', 'text': 'You are a name cl...",3,None
813,"[{'role': 'system', 'text': 'You are a name cl...",3,None
814,"[{'role': 'system', 'text': 'You are a name cl...",3,None


In [12]:
df['response'].value_counts()

response
3          624
1          102
2           59
2\n3        12
3\n3         9
1\n3         4
1\n2         2
2 2 2 2      1
3\n2         1
2\n3\n3      1
1\n3\n3      1
Name: count, dtype: int64

In [17]:
import pandas as pd
import json

# === Пути ===
INPUT_CSV = "/Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_matrix.csv"
RESULTS_PARQUET = "/Users/kseniazavyalova/Downloads/Batch inference fbii29i5t83mpq464gt8 result.parquet"
OUTPUT_CSV = "/Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_classified.csv"

results_df = pd.read_parquet(RESULTS_PARQUET)

print(f"Колонки в parquet: {results_df.columns.tolist()}")
print(f"Строк в parquet: {len(results_df)}")
results_df.head(2)

# Находим нужные колонки
request_col = next((c for c in ['request', 'input', 'prompt'] if c in results_df.columns), None)
response_col = next((c for c in ['response', 'output', 'generated_text', 'completion'] if c in results_df.columns), None)

print(f"Колонка с запросом: {request_col}")
print(f"Колонка с ответом:  {response_col}")

import numpy as np

name_to_response = {}

for idx, row in results_df.iterrows():
    try:
        request_data = row['request']

        # Если это список / ndarray — берём как есть
        if isinstance(request_data, (list, np.ndarray)):
            messages = list(request_data)
        # Если это dict — достаём по ключу 'request'
        elif isinstance(request_data, dict):
            messages = request_data.get('request', [])
        # Если это строка JSON — парсим
        elif isinstance(request_data, str):
            parsed = json.loads(request_data)
            messages = parsed.get('request', []) if isinstance(parsed, dict) else parsed
        else:
            continue

        # Ищем последнее сообщение с ролью user
        user_text = None
        for msg in reversed(messages):
            if isinstance(msg, dict) and msg.get('role') == 'user':
                user_text = msg.get('text', '')
                break

        if not user_text:
            continue

        # Извлекаем response (может быть строкой, dict или ndarray)
        response_data = row['response']
        if isinstance(response_data, (list, np.ndarray)) and len(response_data) > 0:
            response_text = response_data[0]
        elif isinstance(response_data, dict):
            response_text = response_data.get('text', response_data.get('response', str(response_data)))
        else:
            response_text = response_data

        name_to_response[user_text] = response_text

    except Exception as e:
        print(f"Ошибка в строке {idx}: {e}")
        continue

print(f"Извлечено пар имя -> response: {len(name_to_response)}")

# Смотрим несколько примеров, чтобы убедиться что всё ок
list(name_to_response.items())[:10]

input_df = pd.read_csv(INPUT_CSV)
print(f"Строк в исходном CSV: {len(input_df)}")

# Временная колонка в нижнем регистре для мержа
input_df['name_lower'] = input_df['name'].astype(str).str.strip().str.lower()
input_df['classification'] = input_df['name_lower'].map(name_to_response)

matched = input_df['classification'].notna().sum()
print(f"Удалось смержить: {matched} из {len(input_df)}")

# Удаляем вспомогательную колонку
input_df = input_df.drop(columns=['name_lower'])

input_df.head()

input_df.to_csv(OUTPUT_CSV, index=False)

print(f"Сохранено: {OUTPUT_CSV}")
print(f"Всего строк: {len(input_df)}")
print(f"С классификацией: {input_df['classification'].notna().sum()}")
print(f"Без классификации: {input_df['classification'].isna().sum()}")

Колонки в parquet: ['request', 'response', 'error']
Строк в parquet: 816
Колонка с запросом: request
Колонка с ответом:  response
Извлечено пар имя -> response: 816
Строк в исходном CSV: 817
Удалось смержить: 816 из 817
Сохранено: /Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_classified.csv
Всего строк: 817
С классификацией: 816
Без классификации: 1


In [18]:
input_df

,name,bert_base,spacy_sm,stat_propn,xlm_roberta_base,classification
0,NaN,0.656044,0.00,0.00,0.967005,NaN
1,a,0.989385,0.00,0.00,0.999826,3
2,aba,0.616613,0.00,0.00,0.000000,3
3,ac,0.000000,0.00,0.00,0.895790,3
4,ack,0.737923,0.00,0.00,0.000000,3
...,...,...,...,...,...,...
812,zivilyns carpet,0.000000,0.75,0.00,0.000000,3
813,zo,0.668801,0.00,0.00,0.932950,3
814,zolamon,0.000000,0.75,0.65,0.999203,3
815,zram,0.376445,0.00,0.00,0.000000,3


In [19]:
import pandas as pd
import json

# === Пути ===
INPUT_CSV = "/Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_matrix.csv"
RESULTS_PARQUET = "/Users/kseniazavyalova/Downloads/Batch inference fbi7mqk43eml4q76sic2 result.parquet"
OUTPUT_CSV = "/Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_classified.csv"

results_df = pd.read_parquet(RESULTS_PARQUET)

print(f"Колонки в parquet: {results_df.columns.tolist()}")
print(f"Строк в parquet: {len(results_df)}")
results_df.head(2)

# Находим нужные колонки
request_col = next((c for c in ['request', 'input', 'prompt'] if c in results_df.columns), None)
response_col = next((c for c in ['response', 'output', 'generated_text', 'completion'] if c in results_df.columns), None)

print(f"Колонка с запросом: {request_col}")
print(f"Колонка с ответом:  {response_col}")

import numpy as np

name_to_response = {}

for idx, row in results_df.iterrows():
    try:
        request_data = row['request']

        # Если это список / ndarray — берём как есть
        if isinstance(request_data, (list, np.ndarray)):
            messages = list(request_data)
        # Если это dict — достаём по ключу 'request'
        elif isinstance(request_data, dict):
            messages = request_data.get('request', [])
        # Если это строка JSON — парсим
        elif isinstance(request_data, str):
            parsed = json.loads(request_data)
            messages = parsed.get('request', []) if isinstance(parsed, dict) else parsed
        else:
            continue

        # Ищем последнее сообщение с ролью user
        user_text = None
        for msg in reversed(messages):
            if isinstance(msg, dict) and msg.get('role') == 'user':
                user_text = msg.get('text', '')
                break

        if not user_text:
            continue

        # Извлекаем response (может быть строкой, dict или ndarray)
        response_data = row['response']
        if isinstance(response_data, (list, np.ndarray)) and len(response_data) > 0:
            response_text = response_data[0]
        elif isinstance(response_data, dict):
            response_text = response_data.get('text', response_data.get('response', str(response_data)))
        else:
            response_text = response_data

        name_to_response[user_text] = response_text

    except Exception as e:
        print(f"Ошибка в строке {idx}: {e}")
        continue

print(f"Извлечено пар имя -> response: {len(name_to_response)}")

# Смотрим несколько примеров, чтобы убедиться что всё ок
list(name_to_response.items())[:10]

input_df = pd.read_csv(INPUT_CSV)
print(f"Строк в исходном CSV: {len(input_df)}")

# Временная колонка в нижнем регистре для мержа
input_df['name_lower'] = input_df['name'].astype(str).str.strip().str.lower()
input_df['classification'] = input_df['name_lower'].map(name_to_response)

matched = input_df['classification'].notna().sum()
print(f"Удалось смержить: {matched} из {len(input_df)}")

# Удаляем вспомогательную колонку
input_df = input_df.drop(columns=['name_lower'])

input_df.head()

input_df.to_csv(OUTPUT_CSV, index=False)

print(f"Сохранено: {OUTPUT_CSV}")
print(f"Всего строк: {len(input_df)}")
print(f"С классификацией: {input_df['classification'].notna().sum()}")
print(f"Без классификации: {input_df['classification'].isna().sum()}")

Колонки в parquet: ['request', 'response', 'error']
Строк в parquet: 816
Колонка с запросом: request
Колонка с ответом:  response
Извлечено пар имя -> response: 816
Строк в исходном CSV: 817
Удалось смержить: 816 из 817
Сохранено: /Users/kseniazavyalova/Desktop/ner_results_batch/ner_A_Warrior_s_Journey_classified.csv
Всего строк: 817
С классификацией: 816
Без классификации: 1


In [20]:
input_df

,name,bert_base,spacy_sm,stat_propn,xlm_roberta_base,classification
0,NaN,0.656044,0.00,0.00,0.967005,NaN
1,a,0.989385,0.00,0.00,0.999826,3
2,aba,0.616613,0.00,0.00,0.000000,3
3,ac,0.000000,0.00,0.00,0.895790,3
4,ack,0.737923,0.00,0.00,0.000000,3
...,...,...,...,...,...,...
812,zivilyns carpet,0.000000,0.75,0.00,0.000000,3
813,zo,0.668801,0.00,0.00,0.932950,3
814,zolamon,0.000000,0.75,0.65,0.999203,3
815,zram,0.376445,0.00,0.00,0.000000,3


In [22]:
input_df['classification'].value_counts()

classification
3             545
2             171
1              80
2\n3           11
3\n3\n3\n3      1
3\n2            1
2\n2            1
23              1
1\n3            1
3\n3            1
233             1
133             1
2\n3\n3         1
Name: count, dtype: int64

In [23]:
female_names = input_df[input_df['classification'].astype(str) == '1'].sort_values('name')
male_names   = input_df[input_df['classification'].astype(str) == '2'].sort_values('name')

print(f"Женских имён: {len(female_names)}")
female_names

Женских имён: 80


,name,bert_base,spacy_sm,stat_propn,xlm_roberta_base,classification
23,amal,0.875077,0.00,0.0,0.999815,1
30,ana,0.390589,0.00,0.0,0.000000,1
32,anza,0.000000,0.00,0.0,0.937248,1
35,ari,0.986806,0.00,0.0,0.000000,1
41,astarin,0.000000,0.00,0.0,0.952312,1
...,...,...,...,...,...,...
802,yoralyns,0.000000,0.75,0.0,0.000000,1
803,yourekarilkanel,0.000000,0.75,0.0,0.000000,1
806,zabanath,0.000000,0.75,0.0,0.998916,1
808,zan,0.000000,0.00,0.8,0.000000,1


In [24]:
print(f"Мужских имён: {len(male_names)}")
male_names

Мужских имён: 171


,name,bert_base,spacy_sm,stat_propn,xlm_roberta_base,classification
5,ackal,0.862751,0.75,0.80,0.932530,2
9,ackal ii dermount,0.000000,0.00,0.00,0.985947,2
10,ackal iii,0.000000,0.00,0.00,0.999287,2
11,ackal the great,0.000000,0.00,0.00,0.999479,2
27,ambrodel,0.000000,0.00,0.65,0.000000,2
...,...,...,...,...,...,...
797,ylocost,0.000000,0.00,0.00,0.952480,2
798,ymon,0.000000,0.00,0.00,0.942133,2
807,zalay,0.000000,0.00,0.70,0.979537,2
810,zimm,0.000000,0.00,0.80,0.000000,2


In [25]:
import pandas as pd
import json
from pathlib import Path

# === Папки и паттерн имён файлов ===
DATA_DIRS = [
    Path("/Users/kseniazavyalova/Desktop/ner_results_batch"),
    Path("/Users/kseniazavyalova/Desktop/ner_results_batch2"),
]
INPUT_PATTERN = "*_matrix.csv"

# Общий выходной JSONL кладём в первую папку (или можно вынести в отдельное место)
OUTPUT_JSONL = DATA_DIRS[0] / "name_classification_prompts_all_unique.jsonl"

# === Системный промпт ===
SYSTEM_PROMPT = (
    "You are a name classifier for fantasy fiction texts. "
    "All input tokens are already in lowercase — do NOT use "
    "capitalization as a signal.\n\n"
    "For each input token, respond with exactly one digit:\n"
    "  1 — female given name\n"
    "  2 — male given name\n"
    "  3 — not a name (common noun, place, surname, object, "
    "species, abbreviation, etc.)\n\n"
    "Rules:\n"
    "1. Output ONLY the digit (1, 2, or 3).\n"
    "2. No explanations, no punctuation, no extra text.\n"
    "3. Names may be fictional/fantasy names that do not exist "
    "in the real world. Classify them based on whether they "
    "function as a personal given name — not a place, object, "
    "title, or species.\n"
    "4. Ignore capitalization entirely — all inputs are lowercase. "
    "Rely on the lexical and morphological nature of the token, "
    "not its casing.\n"
    "5. If ambiguous, choose the most likely category."
)


def build_request(name: str) -> dict:
    messages = [
        {"role": "system", "text": SYSTEM_PROMPT},
        {"role": "user",   "text": name},
    ]
    return {"request": messages}


unique_names = set()
total_processed = 0
total_failed = 0

for data_dir in DATA_DIRS:
    print(f"\n=== Папка: {data_dir} ===")

    if not data_dir.exists():
        print(f"[SKIP] Папка не существует: {data_dir}")
        continue

    input_files = sorted(data_dir.glob(INPUT_PATTERN))

    if not input_files:
        print(f"Не найдено файлов по паттерну '{INPUT_PATTERN}'")
        continue

    print(f"Найдено файлов: {len(input_files)}")

    processed = 0
    failed = 0

    for input_path in input_files:
        try:
            df = pd.read_csv(input_path)

            if "name" not in df.columns:
                raise KeyError(
                    f"В файле {input_path.name} нет колонки 'name'. "
                    f"Найдены: {df.columns.tolist()}"
                )

            for raw_name in df["name"]:
                if pd.isna(raw_name):
                    continue
                name = str(raw_name).strip().lower()
                if name:
                    unique_names.add(name)

            processed += 1
            print(f"  [OK] {input_path.name}")

        except Exception as e:
            failed += 1
            print(f"  [FAIL] {input_path.name}: {e}")

    print(f"  Итого по папке: обработано {processed}, с ошибкой {failed}")
    total_processed += processed
    total_failed += failed

print(f"\nСобрано уникальных имен (по всем папкам): {len(unique_names)}")

    # Записываем один общий JSONL
records_written = 0

with open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:
    for name in sorted(unique_names):
        request_obj = build_request(name)
        fout.write(json.dumps(request_obj, ensure_ascii=False) + "\n")
        records_written += 1

print(f"\n=== Итог ===")
print(f"Папок обработано: {len(DATA_DIRS)}")
print(f"Файлов обработано успешно: {total_processed}")
print(f"Файлов с ошибкой: {total_failed}")
print(f"Уникальных имен: {len(unique_names)}")
print(f"Записей в JSONL: {records_written}")
print(f"Результат: {OUTPUT_JSONL}")



=== Папка: /Users/kseniazavyalova/Desktop/ner_results_batch ===
Найдено файлов: 33
  [OK] ner_A_Feast_for_Crows_matrix.csv
  [OK] ner_A_Street_Cat_Named_Bob_matrix.csv
  [OK] ner_A_Warrior_s_Journey_matrix.csv
  [OK] ner_BRIDE_matrix.csv
  [OK] ner_CHAPTER_ONE_matrix.csv
  [OK] ner_CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW_matrix.csv
  [OK] ner_City_of_Glass_matrix.csv
  [OK] ner_Fourth_Wing_matrix.csv
  [OK] ner_Gideon_the_Ninth_matrix.csv
  [OK] ner_Girl_Online_matrix.csv
  [OK] ner_Harry_Potter_and_the_Sorcerer_s_Stone_matrix.csv
  [OK] ner_Ice_Station_Zebra_matrix.csv
  [OK] ner_It_ends_with_us_matrix.csv
  [OK] ner_Juniper___Thorn_matrix.csv
  [OK] ner_Mortal_Engines_matrix.csv
  [OK] ner_Realms_of_the_Dragons_vol_1_matrix.csv
  [OK] ner_Roots_of_Chaos_3__Among_the_Burning_Flow_matrix.csv
  [OK] ner_Royal_Fashion_2__All_the_Gossip_From_Par_matrix.csv
  [OK] ner_Sands_of_the_Soul_matrix.csv
  [OK] ner_Tamora_Pierce_-_The_Will_of_the_matrix.csv
  [OK] ner_The_Hobbit_matrix.csv
  [OK

In [26]:
from pathlib import Path

# === Пути ===
SOURCE_JSONL = Path("/Users/kseniazavyalova/Desktop/ner_results_batch/name_classification_prompts_all_unique.jsonl")
OUTPUT_DIR = SOURCE_JSONL.parent
BASE_NAME = SOURCE_JSONL.stem  # "name_classification_prompts_all_unique"

# === Параметры разбиения ===
CHUNK_SIZE = 10_000
NUM_CHUNKS = 5


def main():
    if not SOURCE_JSONL.exists():
        print(f"Файл не найден: {SOURCE_JSONL}")
        return

    # Открываем все 5 выходных файлов сразу
    output_files = []
    for i in range(1, NUM_CHUNKS + 1):
        out_path = OUTPUT_DIR / f"{BASE_NAME}_part{i}.jsonl"
        output_files.append(open(out_path, "w", encoding="utf-8"))

    # Счётчики
    counts = [0] * NUM_CHUNKS
    current_chunk = 0
    total_lines = 0

    try:
        with open(SOURCE_JSONL, "r", encoding="utf-8") as fin:
            for line in fin:
                if not line.strip():
                    continue

                # Если текущий чанк заполнен и мы ещё не на последнем — переключаемся
                if counts[current_chunk] >= CHUNK_SIZE and current_chunk < NUM_CHUNKS - 1:
                    current_chunk += 1

                output_files[current_chunk].write(line)
                counts[current_chunk] += 1
                total_lines += 1

    finally:
        for f in output_files:
            f.close()

    # === Итог ===
    print("=== Итог разбиения ===")
    for i, count in enumerate(counts, start=1):
        out_path = OUTPUT_DIR / f"{BASE_NAME}_part{i}.jsonl"
        print(f"  part{i}: {count:>6} строк  ->  {out_path.name}")
    print(f"\nВсего строк обработано: {total_lines}")
    print(f"Сумма по чанкам:        {sum(counts)}")


if __name__ == "__main__":
    main()

=== Итог разбиения ===
  part1:  10000 строк  ->  name_classification_prompts_all_unique_part1.jsonl
  part2:  10000 строк  ->  name_classification_prompts_all_unique_part2.jsonl
  part3:  10000 строк  ->  name_classification_prompts_all_unique_part3.jsonl
  part4:  10000 строк  ->  name_classification_prompts_all_unique_part4.jsonl
  part5:   9014 строк  ->  name_classification_prompts_all_unique_part5.jsonl

Всего строк обработано: 49014
Сумма по чанкам:        49014


In [27]:
import pandas as pd
import json
import numpy as np
from pathlib import Path

# === Папки с CSV ===
DATA_DIRS = [
    Path("/Users/kseniazavyalova/Desktop/ner_results_batch"),
    Path("/Users/kseniazavyalova/Desktop/ner_results_batch2"),
]
INPUT_PATTERN = "*_matrix.csv"

# Папка с parquet файлами
PARQUET_DIR = Path("/Users/kseniazavyalova/Downloads/результаты")
PARQUET_PATTERN = "*.parquet"

# === Шаг 1: Собираем все parquet файлы в один маппинг ===
print("=== Сбор всех parquet файлов ===\n")

parquet_files = sorted(PARQUET_DIR.glob(PARQUET_PATTERN))
print(f"Найдено parquet файлов: {len(parquet_files)}")

name_to_response = {}

for pq_file in parquet_files:
    try:
        print(f"\nОбрабатываю: {pq_file.name}")
        results_df = pd.read_parquet(pq_file)
        print(f"  Строк: {len(results_df)}")
        
        # Проверяем наличие нужных колонок
        if 'request' not in results_df.columns or 'response' not in results_df.columns:
            print(f"  [SKIP] Нет нужных колонок. Доступны: {results_df.columns.tolist()}")
            continue
        
        # Извлекаем маппинг
        count_before = len(name_to_response)
        
        for idx, row in results_df.iterrows():
            try:
                request_data = row['request']
                
                # Парсим request
                if isinstance(request_data, (list, np.ndarray)):
                    messages = list(request_data)
                elif isinstance(request_data, dict):
                    messages = request_data.get('request', [])
                elif isinstance(request_data, str):
                    parsed = json.loads(request_data)
                    messages = parsed.get('request', []) if isinstance(parsed, dict) else parsed
                else:
                    continue
                
                # Ищем последнее сообщение с ролью user
                user_text = None
                for msg in reversed(messages):
                    if isinstance(msg, dict) and msg.get('role') == 'user':
                        user_text = msg.get('text', '')
                        break
                
                if not user_text:
                    continue
                
                # Извлекаем response
                response_data = row['response']
                if isinstance(response_data, (list, np.ndarray)) and len(response_data) > 0:
                    response_text = response_data[0]
                elif isinstance(response_data, dict):
                    response_text = response_data.get('text', response_data.get('response', str(response_data)))
                else:
                    response_text = response_data
                
                name_to_response[user_text] = response_text
                
            except Exception as e:
                continue
        
        count_after = len(name_to_response)
        print(f"  Добавлено пар: {count_after - count_before}")
        
    except Exception as e:
        print(f"  [FAIL] Ошибка чтения: {e}")

print(f"\n=== Итого собрано уникальных пар имя -> response: {len(name_to_response)} ===\n")

# === Шаг 2: Обрабатываем каждую книгу ===
print("=== Обработка книг ===\n")

total_books_processed = 0
total_books_failed = 0

for data_dir in DATA_DIRS:
    print(f"\n--- Папка: {data_dir} ---")
    
    if not data_dir.exists():
        print(f"[SKIP] Папка не существует")
        continue
    
    input_files = sorted(data_dir.glob(INPUT_PATTERN))
    
    if not input_files:
        print(f"Не найдено файлов по паттерну '{INPUT_PATTERN}'")
        continue
    
    print(f"Найдено CSV файлов: {len(input_files)}")
    
    for input_path in input_files:
        try:
            # Извлекаем имя книги
            stem = input_path.stem
            if stem.startswith("ner_"):
                stem = stem[len("ner_"):]
            if stem.endswith("_matrix"):
                stem = stem[:-len("_matrix")]
            
            output_path = data_dir / f"{stem}_personal_names.csv"
            
            # Читаем CSV
            input_df = pd.read_csv(input_path)
            
            if "name" not in input_df.columns:
                raise KeyError(f"Нет колонки 'name'")
            
            # Создаём временную колонку для мержа
            input_df['name_lower'] = input_df['name'].astype(str).str.strip().str.lower()
            input_df['classification'] = input_df['name_lower'].map(name_to_response)
            
            # Фильтруем только личные имена (1 или 2)
            # Приводим к строке для надёжного сравнения
            input_df['classification_str'] = input_df['classification'].astype(str)
            personal_names = input_df[input_df['classification_str'].isin(['1', '2'])].copy()
            
            # Удаляем вспомогательные колонки
            personal_names = personal_names.drop(columns=['name_lower', 'classification_str'])
            personal_names = personal_names.rename(columns={'classification': 'llm_classification'})
            
            # Сохраняем
            personal_names.to_csv(output_path, index=False)
            
            total_books_processed += 1
            print(f"  [OK] {input_path.name}")
            print(f"       -> {output_path.name} ({len(personal_names)} личных имён из {len(input_df)})")
            
        except Exception as e:
            total_books_failed += 1
            print(f"  [FAIL] {input_path.name}: {e}")

print(f"\n=== Итог ===")
print(f"Книг обработано успешно: {total_books_processed}")
print(f"Книг с ошибкой: {total_books_failed}")
print(f"Всего уникальных имён в маппинге: {len(name_to_response)}")

=== Сбор всех parquet файлов ===

Найдено parquet файлов: 5

Обрабатываю: Batch inference fbi1vi72nltv1cpkg3g4 result.parquet
  Строк: 10000
  Добавлено пар: 10000

Обрабатываю: Batch inference fbie0qboate6iv80g1e6 result.parquet
  Строк: 10000
  Добавлено пар: 10000

Обрабатываю: Batch inference fbil640fbptraccom75e result.parquet
  Строк: 9014
  Добавлено пар: 9014

Обрабатываю: Batch inference fbioaq8i8ne4bhjum9nf result.parquet
  Строк: 10000
  Добавлено пар: 10000

Обрабатываю: Batch inference fbiqavofbkfp0dqaq5s8 result.parquet
  Строк: 10000
  Добавлено пар: 10000

=== Итого собрано уникальных пар имя -> response: 49014 ===

=== Обработка книг ===


--- Папка: /Users/kseniazavyalova/Desktop/ner_results_batch ---
Найдено CSV файлов: 33
  [OK] ner_A_Feast_for_Crows_matrix.csv
       -> A_Feast_for_Crows_personal_names.csv (1740 личных имён из 5097)
  [OK] ner_A_Street_Cat_Named_Bob_matrix.csv
       -> A_Street_Cat_Named_Bob_personal_names.csv (67 личных имён из 246)
  [OK] ner_A_

In [28]:
import pandas as pd
from pathlib import Path

# === Папки с результатами ===
DATA_DIRS = [
    Path("/Users/kseniazavyalova/Desktop/ner_results_batch"),
    Path("/Users/kseniazavyalova/Desktop/ner_results_batch2"),
]
OUTPUT_PATTERN = "*_personal_names.csv"

# Выходной файл
OUTPUT_CSV = Path("/Users/kseniazavyalova/Desktop/ner_results_batch/all_personal_names_combined.csv")


def extract_book_name(file_path: Path) -> str:
    """
    Извлекает имя книги из имени файла.
    A_Warrior_s_Journey_personal_names.csv  ->  A_Warrior_s_Journey
    """
    stem = file_path.stem  # "A_Warrior_s_Journey_personal_names"
    if stem.endswith("_personal_names"):
        stem = stem[:-len("_personal_names")]
    return stem


def main():
    all_dfs = []
    files_processed = 0
    files_failed = 0

    for data_dir in DATA_DIRS:
        print(f"\n--- Папка: {data_dir} ---")

        if not data_dir.exists():
            print(f"[SKIP] Папка не существует")
            continue

        csv_files = sorted(data_dir.glob(OUTPUT_PATTERN))

        if not csv_files:
            print(f"Не найдено файлов по паттерну '{OUTPUT_PATTERN}'")
            continue

        print(f"Найдено файлов: {len(csv_files)}")

        for csv_file in csv_files:
            try:
                book_name = extract_book_name(csv_file)

                # Читаем CSV
                df = pd.read_csv(csv_file)

                # Добавляем колонку с названием книги в начало
                df.insert(0, 'book_name', book_name)

                all_dfs.append(df)
                files_processed += 1

                print(f"  [OK] {csv_file.name} -> book: {book_name} ({len(df)} строк)")

            except Exception as e:
                files_failed += 1
                print(f"  [FAIL] {csv_file.name}: {e}")

    if not all_dfs:
        print("\nНе удалось загрузить ни одного файла")
        return

    # Объединяем все DataFrame
    print(f"\n=== Объединение {len(all_dfs)} файлов ===")
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # Сохраняем результат
    combined_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n=== Итог ===")
    print(f"Файлов обработано успешно: {files_processed}")
    print(f"Файлов с ошибкой: {files_failed}")
    print(f"Всего строк в объединённом CSV: {len(combined_df)}")
    print(f"Результат сохранён: {OUTPUT_CSV}")

    # Статистика по книгам
    print(f"\n=== Распределение по книгам ===")
    book_counts = combined_df['book_name'].value_counts().sort_index()
    for book, count in book_counts.items():
        print(f"  {book}: {count}")


if __name__ == "__main__":
    main()


--- Папка: /Users/kseniazavyalova/Desktop/ner_results_batch ---
Найдено файлов: 33
  [OK] A_Feast_for_Crows_personal_names.csv -> book: A_Feast_for_Crows (1740 строк)
  [OK] A_Street_Cat_Named_Bob_personal_names.csv -> book: A_Street_Cat_Named_Bob (67 строк)
  [OK] A_Warrior_s_Journey_personal_names.csv -> book: A_Warrior_s_Journey (258 строк)
  [OK] BRIDE_personal_names.csv -> book: BRIDE (140 строк)
  [OK] CHAPTER_ONE_personal_names.csv -> book: CHAPTER_ONE (183 строк)
  [OK] CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW_personal_names.csv -> book: CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW (376 строк)
  [OK] City_of_Glass_personal_names.csv -> book: City_of_Glass (218 строк)
  [OK] Fourth_Wing_personal_names.csv -> book: Fourth_Wing (280 строк)
  [OK] Gideon_the_Ninth_personal_names.csv -> book: Gideon_the_Ninth (170 строк)
  [OK] Girl_Online_personal_names.csv -> book: Girl_Online (145 строк)
  [OK] Harry_Potter_and_the_Sorcerer_s_Stone_personal_names.csv -> book: Harry_Potter_and_the_So

In [30]:
df = pd.read_csv("/Users/kseniazavyalova/Desktop/ner_results_batch/all_personal_names_combined.csv")


In [31]:
df

,book_name,name,bert_base,spacy_sm,stat_propn,xlm_roberta_base,llm_classification
0,A_Feast_for_Crows,a arry,0.743796,0.00,0.0,0.000000,2
1,A_Feast_for_Crows,abe,0.398700,0.00,0.0,0.000000,2
2,A_Feast_for_Crows,ace,0.646843,0.00,0.0,0.000000,2
3,A_Feast_for_Crows,addam,0.000000,0.00,0.8,0.568184,2
4,A_Feast_for_Crows,ady,0.000000,0.00,0.0,0.599489,1
...,...,...,...,...,...,...,...
23945,Wuthering_Heights,varrah,0.000000,0.75,0.0,0.000000,1
23946,Wuthering_Heights,varry,0.000000,0.75,0.0,0.000000,2
23947,Wuthering_Heights,wicked ellen,0.000000,0.75,0.0,0.000000,1
23948,Wuthering_Heights,will hathecliff,0.000000,0.75,0.0,0.000000,2


In [32]:
df['llm_classification'].value_counts()

llm_classification
2    15885
1     8065
Name: count, dtype: int64

In [33]:
import pandas as pd
import spacy
import re
from collections import defaultdict
from pathlib import Path

# === Пути ===
COMBINED_CSV = Path("/Users/kseniazavyalova/Desktop/ner_results_batch/all_personal_names_combined.csv")
BOOKS_CSV_1 = Path("/Users/kseniazavyalova/Desktop/all_books_data.csv")
BOOKS_CSV_2 = Path("/Users/kseniazavyalova/Desktop/all_books_data2.csv")
OUTPUT_CSV = Path("/Users/kseniazavyalova/Desktop/ner_results_batch/all_personal_names_with_adjectives.csv")

# === Загружаем spacy один раз ===
print("[INFO] Загрузка spacy...")
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_500_000

# === Загружаем данные ===
combined_df = pd.read_csv(COMBINED_CSV)
print(f"Имен в combined_df: {len(combined_df)}")
print(f"Книг: {combined_df['book_name'].nunique()}")

books_df_1 = pd.read_csv(BOOKS_CSV_1)
books_df_2 = pd.read_csv(BOOKS_CSV_2)
books_df = pd.concat([books_df_1, books_df_2], ignore_index=True)
print(f"Всего книг с текстами: {len(books_df)}")
print(f"Колонки в books_df: {books_df.columns.tolist()}")

/Users/kseniazavyalova/.pyenv/versions/3.11.0/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


[INFO] Загрузка spacy...
Имен в combined_df: 23950
Книг: 108
Всего книг с текстами: 108
Колонки в books_df: ['Title', 'Authors', 'Genre', 'Text']


In [35]:
books_df['Title'].value_counts()

Title
A Feast for Crows                                                                                1
Anne of Green Gables                                                                             1
Moby Dick                                                                                        1
House of Seven Gables                                                                            1
The Scarlet Letter                                                                               1
                                                                                                ..
It ends with us                                                                                  1
Zodiac Academy 4: Shadow Princess: An Academy Bully Romance (Supernatural Bullies and Beasts)    1
Whimbrel House 1 - Keeper of Enchanted Rooms                                                     1
Veronika decides to die                                                                          1
The 

In [37]:
from difflib import SequenceMatcher

# Нормализуем book_name для префиксного сравнения:
# убираем двойные/тройные подчёркивания и висячие подчёркивания
def normalize_for_prefix(s: str) -> str:
    s = re.sub(r"[_\-]+", "_", s)   # любые последовательности _ и - -> одно _
    s = s.strip("_")
    return s.lower()

# Строим индекс: prefix -> список (нормализованный_ключ, оригинальный_book_name)
books_index = {}
for bn in book_name_to_text.keys():
    prefix = normalize_for_prefix(bn)
    books_index.setdefault(prefix, []).append(bn)

# Сопоставляем каждую книгу из combined_df
book_name_to_text_full = dict(book_name_to_text)  # копия
unmatched = []
manual_map = {}

for b in combined_df["book_name"].unique():
    if b in book_name_to_text_full:
        continue  # уже есть

    b_prefix = normalize_for_prefix(b)

    # 1) Ищем точное совпадение по нормализованному префиксу
    candidates = books_index.get(b_prefix, [])

    # 2) Если не нашли — ищем по префиксу (startswith)
    if not candidates:
        candidates = [
            orig for norm, orig in
            [(normalize_for_prefix(k), k) for k in book_name_to_text.keys()]
            if norm.startswith(b_prefix) or b_prefix.startswith(norm)
        ]

    # 3) Если всё ещё нет — ищем близкие по similarity
    if not candidates:
        scored = []
        for k in book_name_to_text.keys():
            k_norm = normalize_for_prefix(k)
            score = SequenceMatcher(None, b_prefix, k_norm).ratio()
            if score > 0.7:
                scored.append((score, k))
        scored.sort(reverse=True)
        candidates = [k for _, k in scored[:3]]

    if candidates:
        # Берём самый короткий (наименее вероятно что это другая книга)
        best = min(candidates, key=len)
        manual_map[b] = best
        book_name_to_text_full[b] = book_name_to_text[best]
    else:
        unmatched.append(b)

# Выводим результат
print(f"=== Сопоставлено вручную: {len(manual_map)} ===")
for k, v in manual_map.items():
    print(f"  {k}")
    print(f"      -> {v}")

print(f"\n=== По-прежнему не нашлись: {len(unmatched)} ===")
for b in unmatched:
    print(f"  - {b}")

print(f"\n=== Итого книг с текстом: {len(book_name_to_text_full)} ===")

=== Сопоставлено вручную: 8 ===
  CRESCENT__CITY_HOUSE_OF_FLAME_AND_SHADOW
      -> CRESCENT_CITY_HOUSE_OF_FLAME_AND_SHADOW
  Juniper___Thorn
      -> Juniper_Thorn
  Roots_of_Chaos_3__Among_the_Burning_Flow
      -> Roots_of_Chaos_3_Among_the_Burning_Flowers
  Royal_Fashion_2__All_the_Gossip_From_Par
      -> Royal_Fashion_2_All_the_Gossip_From_Paris
  Tamora_Pierce_-_The_Will_of_the
      -> Tamora_Pierce_The_Will_of_the
  The_Thief-Taker_s_Blade
      -> The_Thief_Taker_s_Blade
  Whimbrel_House_1_-_Keeper_of_Enchanted_R
      -> Whimbrel_House_1_Keeper_of_Enchanted_Rooms
  Zodiac_Academy_4__Shadow_Princess__An_Ac
      -> Zodiac_Academy_4_Shadow_Princess_An_Academy_Bully_Romance_Supernatural_Bullies_and_Beasts

=== По-прежнему не нашлись: 0 ===

=== Итого книг с текстом: 116 ===


In [38]:
# Заменяем укороченные book_name в combined_df на полные из books_df
combined_df["book_name"] = combined_df["book_name"].map(
    lambda x: manual_map.get(x, x)
)

# Проверяем что задвоений нет
unique_books = combined_df["book_name"].unique()
print(f"Уникальных книг в combined_df после замены: {len(unique_books)}")

# Проверяем покрытие
matched = sum(1 for b in unique_books if b in book_name_to_text)
print(f"Книг с найденным текстом: {matched} / {len(unique_books)}")

# Теперь используем оригинальный book_name_to_text (без _full)
book_name_to_text_final = book_name_to_text

Уникальных книг в combined_df после замены: 108
Книг с найденным текстом: 108 / 108


In [50]:
import spacy
from collections import defaultdict

# Зависимости, по которым «спускаемся» от имени к прилагательным
# amod  — атрибутивное прилагательное: "brave Arya"
# acomp — предикативное через copula: "Arya was brave"
# attr  — атрибутив после linking verb: "Arya seemed brave"
# oprd  — object predicate: "They called her brave"
ADJ_DEPS = {"amod", "acomp", "attr", "oprd"}

# Зависимости, по которым «поднимаемся» от имени к глаголу-связке
# nsubj / nsubjpass — имя является субъектом
COPULA_DEPS = {"nsubj", "nsubjpass"}

# Местоимения
PRONOUN_MALE   = {"he", "him", "his", "himself"}
PRONOUN_FEMALE = {"she", "her", "hers", "herself"}


def normalize_for_match(name: str) -> str:
    s = re.sub(r"[^a-zA-Z\s]", "", str(name)).lower().strip()
    return " ".join(s.split())


def find_name_tokens(doc_sent, names_norm: list[str]):
    """
    Находит токены имён в spaCy-доке предложения.
    Возвращает список (token, name_norm).
    Имена сортируются по убыванию длины, чтобы длинные находились раньше.
    """
    found = []
    used_positions = set()

    # Сортируем по длине — длинные сначала
    for n in sorted(names_norm, key=len, reverse=True):
        if not n:
            continue
        words = n.split()
        # Ищем последовательность токенов, совпадающую с именем
        for i in range(len(doc_sent) - len(words) + 1):
            if any((i + j) in used_positions for j in range(len(words))):
                continue
            span = doc_sent[i:i + len(words)]
            span_text = " ".join(t.text.lower() for t in span)
            if span_text == n:
                found.append((span, n))
                for j in range(len(words)):
                    used_positions.add(i + j)
                break
    return found


def find_pronoun_tokens(doc_sent):
    """Возвращает список (token, gender) для местоимений."""
    res = []
    for token in doc_sent:
        t_lower = token.text.lower()
        if t_lower in PRONOUN_MALE:
            res.append((token, "M"))
        elif t_lower in PRONOUN_FEMALE:
            res.append((token, "F"))
    return res

def collect_adjectives_for_token(head_token):
    """
    Собирает все прилагательные, синтаксически связанные с head_token.
    """
    adjectives = []
    seen = set()

    def add_adj(token):
        if token.pos_ == "ADJ" and token.i not in seen:
            adjectives.append(token.text.lower())
            seen.add(token.i)
            # Конъюнкты: "brave and fearless"
            for gc in token.children:
                if gc.dep_ == "conj" and gc.pos_ == "ADJ" and gc.i not in seen:
                    adjectives.append(gc.text.lower())
                    seen.add(gc.i)

    # 1. Прямые зависимые с ADJ_DEPS
    for child in head_token.children:
        if child.dep_ in ADJ_DEPS:
            add_adj(child)

    # 2. Если head_token — субъект (nsubj), идём к глаголу-связке
    if head_token.dep_ in COPULA_DEPS:
        verb = head_token.head
        for v_child in verb.children:
            # a) Прямое прилагательное: "Arya was brave"
            if v_child.dep_ in ADJ_DEPS:
                add_adj(v_child)
            # b) Именное сказуемое: "Arya was brave warrior"
            #    warriors (attr/acomp) -> brave (amod)
            elif v_child.dep_ in ("attr", "acomp") and v_child.pos_ in ("NOUN", "PROPN"):
                for noun_child in v_child.children:
                    if noun_child.dep_ == "amod":
                        add_adj(noun_child)

    # 3. xcomp: "Arya seemed brave"
    for child in head_token.children:
        if child.dep_ == "xcomp":
            add_adj(child)

    # 4. Предикативные аппозитивы: "Arya, brave and silent, walked..."
    #    В spaCy это обычно appós -> ADJ
    for child in head_token.children:
        if child.dep_ == "appos" and child.pos_ == "ADJ":
            add_adj(child)

    return adjectives
    
def resolve_coref_by_dependency(
    doc_sent,
    name_tokens,        # [(span, name_norm), ...]
    pronoun_tokens,     # [(token, gender), ...]
    name_to_gender,
    last_mentioned
):
    """
    Возвращает:
      - present_names: {name_norm: [adj1, adj2, ...]} — имена с их прилагательными
      - обновляет last_mentioned
    """
    result = defaultdict(list)

    # 1. Для каждого прямого упоминания имени — собираем его прилагательные
    for span, name_norm in name_tokens:
        # Берём главный токен span (обычно последний — для "Mary Ann" это "Ann")
        head_token = span[-1]
        adjs = collect_adjectives_for_token(head_token)
        result[name_norm].extend(adjs)

    # 2. Для местоимений — находим имя того же пола и приписываем прилагательные местоимения ему
    for pron_token, p_gender in pronoun_tokens:
        # Кандидаты: имена того же пола
        candidates = [n for n, g in last_mentioned.items() if g == p_gender]
        # Плюс имена из этого предложения того же пола
        for _, n in name_tokens:
            if name_to_gender.get(n) == p_gender:
                candidates.append(n)

        if not candidates:
            continue

        chosen = candidates[-1]  # последнее упомянутое

        # Собираем прилагательные, связанные с местоимением
        adjs = collect_adjectives_for_token(pron_token)
        result[chosen].extend(adjs)

    # 3. Обновляем last_mentioned
    for span, name_norm in name_tokens:
        g = name_to_gender.get(name_norm)
        if g in ("1", "2"):
            last_mentioned[name_norm] = "M" if g == "2" else "F"

    return result

In [45]:
def process_book(text: str, names_for_book: pd.DataFrame) -> dict[str, dict]:
    """
    Для данной книги и списка её имён возвращает:
      { name_norm: {"adjectives": {adj: count}, "total_adjectives": int} }
    """
    name_to_gender = {}
    names_norm = []
    for _, row in names_for_book.iterrows():
        n = normalize_for_match(row["name"])
        if not n:
            continue
        names_norm.append(n)
        cls = str(row.get("llm_classification", ""))
        if cls == "1":
            name_to_gender[n] = "F"
        elif cls == "2":
            name_to_gender[n] = "M"
        else:
            name_to_gender[n] = None

    names_norm = sorted(set(names_norm), key=len, reverse=True)

    CHUNK = 500_000
    pieces = [text[i:i+CHUNK] for i in range(0, len(text), CHUNK)]

    adj_counts = defaultdict(lambda: defaultdict(int))
    total_counts = defaultdict(int)
    last_mentioned = {}

    for piece in pieces:
        try:
            doc = nlp(piece)
        except Exception as e:
            print(f"  [WARN] spacy error on chunk: {e}")
            continue

        for sent in doc.sents:
            name_tokens = find_name_tokens(sent, names_norm)
            pronoun_tokens = find_pronoun_tokens(sent)

            if not name_tokens and not pronoun_tokens:
                continue

            result = resolve_coref_by_dependency(
                sent, name_tokens, pronoun_tokens,
                name_to_gender, last_mentioned
            )

            for name_norm, adjs in result.items():
                for a in adjs:
                    adj_counts[name_norm][a] += 1
                    total_counts[name_norm] += 1

    return {
        n: {
            "adjectives": dict(adj_counts[n]),
            "total_adjectives": total_counts[n],
        }
        for n in names_norm
    }

In [43]:
# === Тест на одной книге ===
TEST_BOOK = "Veronika_decides_to_die"

text = book_name_to_text_final[TEST_BOOK]
names_for_book = combined_df[combined_df["book_name"] == TEST_BOOK].copy()

print(f"Книга: {TEST_BOOK}")
print(f"Длина текста: {len(text)} символов")
print(f"Имен в книге: {len(names_for_book)}")

# Запускаем обработку
print(f"\n[INFO] Обработка книги...")
result = process_book(text, names_for_book)

# Показываем результаты
results_list = []
for idx, row in names_for_book.iterrows():
    n = normalize_for_match(row["name"])
    if n in result:
        adj_dict = result[n]["adjectives"]
        total = result[n]["total_adjectives"]
        if total > 0:
            top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
            top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
            results_list.append({
                "name": row["name"],
                "llm_classification": row["llm_classification"],
                "total_adjectives": total,
                "top_adjectives": top_str
            })

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values("total_adjectives", ascending=False)

print(f"\n=== Результаты для {TEST_BOOK} ===")
print(f"Имена с прилагательными (топ-20 по частоте):")
display(results_df.head(20))

print(f"\nВсего имён с прилагательными: {len(results_df)}")
print(f"Суммарное количество прилагательных: {results_df['total_adjectives'].sum()}")

Книга: Veronika_decides_to_die
Длина текста: 289512 символов
Имен в книге: 52

[INFO] Обработка книги...

=== Результаты для Veronika_decides_to_die ===
Имена с прилагательными (топ-20 по частоте):


,name,llm_classification,total_adjectives,top_adjectives
25,veronika,1,231,"other:8, first:6, good:5, afraid:5, happy:4"
6,eduard,2,187,"other:7, new:5, long:5, more:4, normal:4"
27,villete,1,136,"only:7, same:6, young:5, other:5, insane:3"
13,igor,2,116,"more:5, little:4, young:4, such:3, astral:3"
16,mari,1,115,"long:5, right:4, first:3, other:3, sure:3"
28,zedka,1,82,"normal:3, many:3, better:3, little:3, astral:3"
21,paulo,2,17,"particular:1, brazilian:1, algerian:1, sloveni..."
22,paulo coelho,2,14,"particular:1, brazilian:1, algerian:1, sloveni..."
5,coelho,2,14,"particular:1, brazilian:1, algerian:1, sloveni..."
17,maria,1,14,"strange:2, positive:2, slightest:1, exotic:1, ..."



Всего имён с прилагательными: 29
Суммарное количество прилагательных: 1014


In [52]:
# === Тест на одной книге ===
TEST_BOOK = "Veronika_decides_to_die"

text = book_name_to_text_final[TEST_BOOK]
names_for_book = combined_df[combined_df["book_name"] == TEST_BOOK].copy()

print(f"Книга: {TEST_BOOK}")
print(f"Длина текста: {len(text)} символов")
print(f"Имен в книге: {len(names_for_book)}")

# Запускаем обработку
print(f"\n[INFO] Обработка книги...")
result = process_book(text, names_for_book)

# Показываем результаты
results_list = []
for idx, row in names_for_book.iterrows():
    n = normalize_for_match(row["name"])
    if n in result:
        adj_dict = result[n]["adjectives"]
        total = result[n]["total_adjectives"]
        if total > 0:
            top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
            top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
            results_list.append({
                "name": row["name"],
                "llm_classification": row["llm_classification"],
                "total_adjectives": total,
                "top_adjectives": top_str
            })

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values("total_adjectives", ascending=False)

print(f"\n=== Результаты для {TEST_BOOK} ===")
print(f"Имена с прилагательными (топ-20 по частоте):")
display(results_df.head(20))

print(f"\nВсего имён с прилагательными: {len(results_df)}")
print(f"Суммарное количество прилагательных: {results_df['total_adjectives'].sum()}")

Книга: Veronika_decides_to_die
Длина текста: 289512 символов
Имен в книге: 52

[INFO] Обработка книги...

=== Результаты для Veronika_decides_to_die ===
Имена с прилагательными (топ-20 по частоте):


,name,llm_classification,total_adjectives,top_adjectives
5,veronika,1,19,"happy:2, sure:2, afraid:2, interested:1, aware:1"
1,eduard,2,18,"capable:2, agitated:2, sleepy:2, schizophrenic..."
3,mari,1,15,"right:2, certain:2, tired:1, glad:1, sure:1"
2,igor,2,9,"worried:2, ready:1, happy:1, silent:1, romantic:1"
8,zedka,1,8,"serious:1, crazier:1, welcome:1, sound:1, asle..."
7,villete,1,6,"bad:1, crazier:1, liberal:1, different:1, crazy:1"
6,veronikas,1,2,other:2
0,adam,2,1,bored:1
4,nasrudin,2,1,sober:1



Всего имён с прилагательными: 9
Суммарное количество прилагательных: 79


In [54]:
from tqdm import tqdm

# Инициализируем новые колонки
combined_df["adjectives"] = None
combined_df["total_adjectives"] = 0

books_in_df = combined_df["book_name"].unique()
processed = 0
skipped = 0

# Предварительно фильтруем книги, которые реально будем обрабатывать
# (чтобы tqdm показывал корректный total)
books_to_process = []
for book_name in books_in_df:
    if book_name not in book_name_to_text_final:
        skipped += 1
        continue
    text = book_name_to_text_final[book_name]
    if not text or len(text) < 100:
        skipped += 1
        continue
    books_to_process.append((book_name, text))

print(f"Книг к обработке: {len(books_to_process)}, пропущено: {skipped}\n")

pbar = tqdm(books_to_process, desc="Обработка книг", unit="book")

for book_name, text in pbar:
    # Обновляем подпись прогресс-бара
    names_for_book = combined_df[combined_df["book_name"] == book_name]
    pbar.set_postfix({
        "book": book_name[:30],
        "names": len(names_for_book),
        "chars": f"{len(text)//1000}k",
    })

    try:
        result = process_book(text, names_for_book)
    except Exception as e:
        pbar.write(f"  [FAIL] {book_name}: {e}")
        skipped += 1
        continue

    # Записываем обратно в combined_df
    for idx, row in names_for_book.iterrows():
        n = normalize_for_match(row["name"])
        if n in result:
            combined_df.at[idx, "adjectives"] = result[n]["adjectives"] or {}
            combined_df.at[idx, "total_adjectives"] = result[n]["total_adjectives"]

    processed += 1

    # === ОТЛАДКА: показываем пример для этой книги ===
    # Находим имя с максимальным total_adjectives
    names_with_adjs = names_for_book[names_for_book["total_adjectives"] > 0]
    if not names_with_adjs.empty:
        top_name = names_with_adjs.loc[names_with_adjs["total_adjectives"].idxmax()]
        name_str = top_name["name"]
        total = top_name["total_adjectives"]
        adj_dict = top_name["adjectives"]
        # Топ-3 прилагательных
        top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:3]
        top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
        pbar.write(f"  [OK] {book_name[:40]} | пример: '{name_str}' | total={total} | top: {top_str}")
    else:
        pbar.write(f"  [OK] {book_name[:40]} | нет прилагательных")

pbar.close()

print(f"\nГотово. Обработано книг: {processed}, пропущено: {skipped}")
print(f"Строк с прилагательными: {(combined_df['total_adjectives'] > 0).sum()}")

Книг к обработке: 108, пропущено: 0



Обработка книг:   1%|          | 1/108 [30:13<53:53:35, 1813.23s/book, book=A_Street_Cat_Named_Bob, names=67, chars=338k]

  [OK] A_Feast_for_Crows | нет прилагательных


Обработка книг:   2%|▏         | 2/108 [30:42<22:28:58, 763.58s/book, book=A_Warrior_s_Journey, names=258, chars=687k]   

  [OK] A_Street_Cat_Named_Bob | нет прилагательных


Обработка книг:   2%|▏         | 2/108 [32:24<28:37:39, 972.26s/book, book=A_Warrior_s_Journey, names=258, chars=687k]


KeyboardInterrupt: 

In [57]:
combined_df

,book_name,name,bert_base,spacy_sm,stat_propn,xlm_roberta_base,llm_classification,adjectives,total_adjectives
0,A_Feast_for_Crows,a arry,0.743796,0.00,0.0,0.000000,2,{},0
1,A_Feast_for_Crows,abe,0.398700,0.00,0.0,0.000000,2,{},0
2,A_Feast_for_Crows,ace,0.646843,0.00,0.0,0.000000,2,{},0
3,A_Feast_for_Crows,addam,0.000000,0.00,0.8,0.568184,2,{},0
4,A_Feast_for_Crows,ady,0.000000,0.00,0.0,0.599489,1,{},0
...,...,...,...,...,...,...,...,...,...
23945,Wuthering_Heights,varrah,0.000000,0.75,0.0,0.000000,1,None,0
23946,Wuthering_Heights,varry,0.000000,0.75,0.0,0.000000,2,None,0
23947,Wuthering_Heights,wicked ellen,0.000000,0.75,0.0,0.000000,1,None,0
23948,Wuthering_Heights,will hathecliff,0.000000,0.75,0.0,0.000000,2,None,0


In [58]:
# === Тест на одной книге ===
TEST_BOOK = "Veronika_decides_to_die"

text = book_name_to_text_final[TEST_BOOK]
names_for_book = combined_df[combined_df["book_name"] == TEST_BOOK].copy()

print(f"Книга: {TEST_BOOK}")
print(f"Длина текста: {len(text)} символов")
print(f"Имен в книге: {len(names_for_book)}")

# Запускаем обработку
print(f"\n[INFO] Обработка книги...")
result = process_book(text, names_for_book)

# Показываем результаты
results_list = []
for idx, row in names_for_book.iterrows():
    n = normalize_for_match(row["name"])
    if n in result:
        adj_dict = result[n]["adjectives"]
        total = result[n]["total_adjectives"]
        if total > 0:
            top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
            top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
            results_list.append({
                "name": row["name"],
                "llm_classification": row["llm_classification"],
                "total_adjectives": total,
                "top_adjectives": top_str
            })

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values("total_adjectives", ascending=False)

print(f"\n=== Результаты для {TEST_BOOK} ===")
print(f"Имена с прилагательными (топ-20 по частоте):")
display(results_df.head(20))

print(f"\nВсего имён с прилагательными: {len(results_df)}")
print(f"Суммарное количество прилагательных: {results_df['total_adjectives'].sum()}")

Книга: Veronika_decides_to_die
Длина текста: 289512 символов
Имен в книге: 52

[INFO] Обработка книги...

=== Результаты для Veronika_decides_to_die ===
Имена с прилагательными (топ-20 по частоте):


,name,llm_classification,total_adjectives,top_adjectives
5,veronika,1,19,"happy:2, sure:2, afraid:2, interested:1, aware:1"
1,eduard,2,18,"capable:2, agitated:2, sleepy:2, schizophrenic..."
3,mari,1,15,"right:2, certain:2, tired:1, glad:1, sure:1"
2,igor,2,9,"worried:2, ready:1, happy:1, silent:1, romantic:1"
8,zedka,1,8,"serious:1, crazier:1, welcome:1, sound:1, asle..."
7,villete,1,6,"bad:1, crazier:1, liberal:1, different:1, crazy:1"
6,veronikas,1,2,other:2
0,adam,2,1,bored:1
4,nasrudin,2,1,sober:1



Всего имён с прилагательными: 9
Суммарное количество прилагательных: 79


In [61]:
# === Тест на одной книге ===
TEST_BOOK = "A_Street_Cat_Named_Bob"

text = book_name_to_text_final[TEST_BOOK]
names_for_book = combined_df[combined_df["book_name"] == TEST_BOOK].copy()

print(f"Книга: {TEST_BOOK}")
print(f"Длина текста: {len(text)} символов")
print(f"Имен в книге: {len(names_for_book)}")

# Запускаем обработку
print(f"\n[INFO] Обработка книги...")
result = process_book(text, names_for_book)

# Показываем результаты
results_list = []
for idx, row in names_for_book.iterrows():
    n = normalize_for_match(row["name"])
    if n in result:
        adj_dict = result[n]["adjectives"]
        total = result[n]["total_adjectives"]
        if total > 0:
            top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
            top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
            results_list.append({
                "name": row["name"],
                "llm_classification": row["llm_classification"],
                "total_adjectives": total,
                "top_adjectives": top_str
            })

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values("total_adjectives", ascending=False)

print(f"\n=== Результаты для {TEST_BOOK} ===")
print(f"Имена с прилагательными (топ-20 по частоте):")
display(results_df.head(20))

print(f"\nВсего имён с прилагательными: {len(results_df)}")
print(f"Суммарное количество прилагательных: {results_df['total_adjectives'].sum()}")

Книга: A_Street_Cat_Named_Bob
Длина текста: 338873 символов
Имен в книге: 67

[INFO] Обработка книги...

=== Результаты для A_Street_Cat_Named_Bob ===
Имена с прилагательными (топ-20 по частоте):


,name,llm_classification,total_adjectives,top_adjectives
1,bob,2,26,"right:3, old:2, fine:2, smart:2, brighter:1"
0,angel,1,3,"different:1, busy:1, alive:1"
7,tom,2,2,"curious:1, large:1"
2,dylan,2,1,fine:1
3,kurt cobain,2,1,next:1
4,nick,2,1,better:1
5,rosemary,1,1,good:1
6,stan,2,1,latter:1



Всего имён с прилагательными: 8
Суммарное количество прилагательных: 36


In [62]:
# === Статистика по книгам: сортировка по длине текста ===

book_stats = []
for book_name, text in book_name_to_text_final.items():
    names_count = len(combined_df[combined_df["book_name"] == book_name])
    book_stats.append({
        "book_name": book_name,
        "text_length": len(text),
        "names_count": names_count,
    })

stats_df = pd.DataFrame(book_stats)
stats_df = stats_df.sort_values("text_length", ascending=False).reset_index(drop=True)

# Добавляем порядковый номер
stats_df.insert(0, "rank", range(1, len(stats_df) + 1))

# Вывод
print(f"Всего книг: {len(stats_df)}")
print(f"Суммарная длина текстов: {stats_df['text_length'].sum():,} символов")
print(f"Средняя длина книги: {stats_df['text_length'].mean():,.0f} символов")
print(f"Медианная длина: {stats_df['text_length'].median():,.0f} символов")
print(f"\nМинимум: {stats_df['text_length'].min():,} ({stats_df.loc[stats_df['text_length'].idxmin(), 'book_name']})")
print(f"Максимум: {stats_df['text_length'].max():,} ({stats_df.loc[stats_df['text_length'].idxmax(), 'book_name']})")

print(f"\n=== Все книги, отсортированные по длине текста ===")
display(stats_df)

Всего книг: 108
Суммарная длина текстов: 67,755,640 символов
Средняя длина книги: 627,367 символов
Медианная длина: 550,500 символов

Минимум: 21,238 (Shirley)
Максимум: 1,978,971 (Bleak_House)

=== Все книги, отсортированные по длине текста ===


,rank,book_name,text_length,names_count
0,1,Bleak_House,1978971,278
1,2,David_Copperfield,1972851,285
2,3,Tom_Jones,1960956,368
3,4,Middlemarch,1780107,504
4,5,Vanity_Fair,1745856,804
...,...,...,...,...
103,104,Bartleby_the_Scrivener,83398,18
104,105,Adam_Bede,80728,123
105,106,The_Castle_of_Otranto,50731,7
106,107,A_Country_Doctor,40846,12


In [64]:
# === Отладочный цикл на 10 самых коротких книгах ===

# 1. Собираем статистику по книгам и сортируем по длине
book_lengths = []
for book_name, text in book_name_to_text_final.items():
    names_count = len(combined_df[combined_df["book_name"] == book_name])
    book_lengths.append((book_name, len(text), names_count))

book_lengths.sort(key=lambda x: x[1])  # по возрастанию длины

# 2. Берём 10 самых коротких
TOP_N = 40
books_to_test = book_lengths[:TOP_N]

print(f"=== Топ-{TOP_N} самых коротких книг ===")
for i, (bn, length, names_count) in enumerate(books_to_test, 1):
    print(f"  {i:2d}. {bn[:50]:<50} | {length:>8,} симв. | {names_count:>4} имён")

print()

# 3. Создаём новый датасет для результатов
all_results = []

# 4. Цикл по книгам
for i, (book_name, text_len, names_count) in enumerate(books_to_test, 1):
    print(f"\n[{i}/{TOP_N}] === {book_name} ===")
    print(f"  Длина текста: {text_len:,} символов")
    print(f"  Имен в книге: {names_count}")

    text = book_name_to_text_final[book_name]
    names_for_book = combined_df[combined_df["book_name"] == book_name].copy()

    # Обработка
    print(f"  [INFO] Обработка...")
    try:
        result = process_book(text, names_for_book)
    except Exception as e:
        print(f"  [FAIL] {e}")
        continue

    # Собираем результаты
    book_results = []
    for idx, row in names_for_book.iterrows():
        n = normalize_for_match(row["name"])
        if n in result:
            adj_dict = result[n]["adjectives"]
            total = result[n]["total_adjectives"]
            if total > 0:
                top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
                top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
                book_results.append({
                    "book_name": book_name,
                    "name": row["name"],
                    "llm_classification": row["llm_classification"],
                    "total_adjectives": total,
                    "top_adjectives": top_str,
                    "adjectives": adj_dict,
                })

    # Добавляем в общий список
    all_results.extend(book_results)

    # Отладка: пример для этой книги
    if book_results:
        top_example = max(book_results, key=lambda x: x["total_adjectives"])
        print(f"  [OK] пример: '{top_example['name']}' | total={top_example['total_adjectives']} | top: {top_example['top_adjectives']}")
    else:
        print(f"  [OK] нет прилагательных")
    print(f"  Найдено имён с прилагательными: {len(book_results)}")

# 5. Формируем итоговый DataFrame
results_df = pd.DataFrame(all_results)

print(f"\n{'='*60}")
print(f"=== ИТОГ ===")
print(f"Всего записей: {len(results_df)}")
if not results_df.empty:
    print(f"Книг с данными: {results_df['book_name'].nunique()}")
    print(f"Суммарное количество прилагательных: {results_df['total_adjectives'].sum():,}")
    print(f"\nРаспределение по книгам:")
    display(results_df.groupby('book_name').agg(
        names_count=('name', 'count'),
        total_adjs=('total_adjectives', 'sum')
    ).sort_values('total_adjs', ascending=False))

    print(f"\n=== Топ-20 имён по частоте прилагательных ===")
    display(results_df.sort_values('total_adjectives', ascending=False).head(20))
else:
    print("Нет данных для отображения.")

=== Топ-40 самых коротких книг ===
   1. Shirley                                            |   21,238 симв. |    6 имён
   2. A_Country_Doctor                                   |   40,846 симв. |   12 имён
   3. The_Castle_of_Otranto                              |   50,731 симв. |    7 имён
   4. Adam_Bede                                          |   80,728 симв. |  123 имён
   5. Bartleby_the_Scrivener                             |   83,398 симв. |   18 имён
   6. The_Thief_Taker_s_Blade                            |   84,255 симв. |   41 имён
   7. Importance_of_Being_Earnest                        |  117,286 симв. |   71 имён
   8. The_Monk                                           |  122,294 симв. |   72 имён
   9. Daisy_Miller                                       |  124,937 симв. |   28 имён
  10. Mary_A_Fiction                                     |  148,866 симв. |   17 имён
  11. The_House_of_Mirth                                 |  153,601 симв. |  203 имён
  12. Agnes_Grey   

,names_count,total_adjs
book_name,,
Harry_Potter_and_the_Sorcerer_s_Stone,28,382
Prince_and_the_Pauper,28,293
Mortal_Engines,26,292
Silas_Marner,26,253
The_Mage_In_The_Iron_Mask,23,213
Roots_of_Chaos_3_Among_the_Burning_Flowers,44,193
The_Story_of_an_Hour,34,135
The_Invisible_Man,6,132
The_Yellow_Wallpaper,20,128



=== Топ-20 имён по частоте прилагательных ===


,book_name,name,llm_classification,total_adjectives,top_adjectives,adjectives
577,Harry_Potter_and_the_Sorcerer_s_Stone,he,2,249,"sure:19, able:9, tall:8, busy:6, nervous:6","{'stupid': 3, 'sure': 19, 'worried': 1, 'upset..."
421,Prince_and_the_Pauper,he,2,213,"glad:10, able:10, mad:10, deep:6, awake:6","{'ruffian': 1, 'handed': 3, 'glad': 10, 'deep'..."
525,Mortal_Engines,he,2,200,"right:12, sure:11, able:11, dead:8, sorry:7","{'bored': 2, 'sorry': 7, 'bent': 2, 'lucky': 2..."
549,The_Mage_In_The_Iron_Mask,he,2,134,"able:11, about:10, more:9, dead:7, sure:7","{'able': 11, 'free': 1, 'conscious': 3, 'awake..."
374,Silas_Marner,he,2,124,"sure:6, silent:6, likely:5, ill:4, unable:4","{'ill': 4, 'worth': 1, 'faultless': 2, 'asleep..."
203,The_Invisible_Man,he,2,121,"aware:9, afraid:5, mad:5, alone:4, black:4","{'afraid': 5, 'bad': 1, 'sensitive': 2, 'audib..."
491,The_Story_of_an_Hour,he,2,87,"untruthful:10, romantic:6, satisfied:5, able:5...","{'worse': 1, 'content': 2, 'stricter': 1, 'unt..."
232,The_Yellow_Wallpaper,he,2,71,"able:6, unable:5, dead:4, asleep:4, glad:3","{'able': 6, 'agitated': 2, 'glad': 3, 'young':..."
339,Roots_of_Chaos_3_Among_the_Burning_Flowers,he,2,69,"cold:4, able:4, asleep:4, angry:3, hale:2","{'hale': 2, 'well': 2, 'sovereign': 2, 'full':..."
154,The_Magician_s_Nephew,he,2,57,"sure:5, mad:4, interested:3, right:3, able:3","{'mad': 4, 'exciting': 2, 'interested': 3, 'dr..."


In [65]:
OUTPUT_DIR = Path("/Users/kseniazavyalova/Desktop/ner_results_batch")
OUTPUT_TEST_CSV = OUTPUT_DIR / "test_short_books_adjectives.csv"

if not results_df.empty:
    # Сохраняем CSV
    results_df.to_csv(OUTPUT_TEST_CSV, index=False)
    print(f"\n💾 Сохранено: {OUTPUT_TEST_CSV}")
    print(f"   Строк: {len(results_df)}")
    print(f"   Размер файла: {OUTPUT_TEST_CSV.stat().st_size / 1024:.1f} KB")

    # Бонус: сохраняем отдельно JSON с полным словарём прилагательных
    # (удобно для последующего чтения без парсинга str-представления dict)
    OUTPUT_TEST_JSON = OUTPUT_DIR / "test_short_books_adjectives.json"
    results_df.to_json(OUTPUT_TEST_JSON, orient="records", force_ascii=False, indent=2)
    print(f"💾 Сохранено (JSON): {OUTPUT_TEST_JSON}")
else:
    print("\n[WARN] Нечего сохранять — results_df пустой")


💾 Сохранено: /Users/kseniazavyalova/Desktop/ner_results_batch/test_short_books_adjectives.csv
   Строк: 593
   Размер файла: 61.6 KB
💾 Сохранено (JSON): /Users/kseniazavyalova/Desktop/ner_results_batch/test_short_books_adjectives.json


In [66]:
# === Следующие 40 книг (с 41 по 80 по длине) ===

from pathlib import Path

# 1. Собираем статистику по книгам и сортируем по длине
book_lengths = []
for book_name, text in book_name_to_text_final.items():
    names_count = len(combined_df[combined_df["book_name"] == book_name])
    book_lengths.append((book_name, len(text), names_count))

book_lengths.sort(key=lambda x: x[1])  # по возрастанию длины

# 2. Берём следующие 40 книг (с 40 по 80)
START_IDX = 40
END_IDX = 80
books_to_test = book_lengths[START_IDX:END_IDX]

print(f"=== Книги {START_IDX+1}-{min(END_IDX, len(book_lengths))} по длине ===")
for i, (bn, length, names_count) in enumerate(books_to_test, START_IDX + 1):
    print(f"  {i:2d}. {bn[:50]:<50} | {length:>8,} симв. | {names_count:>4} имён")

print()

# 3. Создаём новый датасет для результатов
all_results = []

# 4. Цикл по книгам
for i, (book_name, text_len, names_count) in enumerate(books_to_test, START_IDX + 1):
    print(f"\n[{i}/{END_IDX}] === {book_name} ===")
    print(f"  Длина текста: {text_len:,} символов")
    print(f"  Имен в книге: {names_count}")

    text = book_name_to_text_final[book_name]
    names_for_book = combined_df[combined_df["book_name"] == book_name].copy()

    # Обработка
    print(f"  [INFO] Обработка...")
    try:
        result = process_book(text, names_for_book)
    except Exception as e:
        print(f"  [FAIL] {e}")
        continue

    # Собираем результаты
    book_results = []
    for idx, row in names_for_book.iterrows():
        n = normalize_for_match(row["name"])
        if n in result:
            adj_dict = result[n]["adjectives"]
            total = result[n]["total_adjectives"]
            if total > 0:
                top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
                top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
                book_results.append({
                    "book_name": book_name,
                    "name": row["name"],
                    "llm_classification": row["llm_classification"],
                    "total_adjectives": total,
                    "top_adjectives": top_str,
                    "adjectives": adj_dict,
                })

    # Добавляем в общий список
    all_results.extend(book_results)

    # Отладка: пример для этой книги
    if book_results:
        top_example = max(book_results, key=lambda x: x["total_adjectives"])
        print(f"  [OK] пример: '{top_example['name']}' | total={top_example['total_adjectives']} | top: {top_example['top_adjectives']}")
    else:
        print(f"  [OK] нет прилагательных")
    print(f"  Найдено имён с прилагательными: {len(book_results)}")

# 5. Формируем DataFrame новых результатов
new_results_df = pd.DataFrame(all_results)

print(f"\n{'='*60}")
print(f"=== ИТОГ для книг {START_IDX+1}-{END_IDX} ===")
print(f"Всего новых записей: {len(new_results_df)}")
if not new_results_df.empty:
    print(f"Книг с данными: {new_results_df['book_name'].nunique()}")
    print(f"Суммарное количество прилагательных: {new_results_df['total_adjectives'].sum():,}")

# 6. === ДОПИСЫВАЕМ В СУЩЕСТВУЮЩИЙ ФАЙЛ ===
OUTPUT_DIR = Path("/Users/kseniazavyalova/Desktop/ner_results_batch")
OUTPUT_TEST_CSV = OUTPUT_DIR / "test_short_books_adjectives.csv"
OUTPUT_TEST_JSON = OUTPUT_DIR / "test_short_books_adjectives.json"

if not new_results_df.empty:
    # Читаем существующий CSV
    if OUTPUT_TEST_CSV.exists():
        existing_df = pd.read_csv(OUTPUT_TEST_CSV)
        print(f"\n📖 Прочитан существующий файл: {len(existing_df)} строк")
        
        # Конкатенируем
        combined_results_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    else:
        print(f"\n[WARN] Файл не найден, создаём новый")
        combined_results_df = new_results_df
    
    # Сохраняем CSV
    combined_results_df.to_csv(OUTPUT_TEST_CSV, index=False)
    print(f"\n💾 Сохранено (CSV):  {OUTPUT_TEST_CSV}")
    print(f"   Всего строк: {len(combined_results_df)}")
    print(f"   Размер: {OUTPUT_TEST_CSV.stat().st_size / 1024:.1f} KB")

    # JSON
    combined_results_df.to_json(OUTPUT_TEST_JSON, orient="records", force_ascii=False, indent=2)
    print(f"💾 Сохранено (JSON): {OUTPUT_TEST_JSON}")
    print(f"   Размер: {OUTPUT_TEST_JSON.stat().st_size / 1024:.1f} KB")
else:
    print("\n[WARN] Нет новых данных для сохранения")

=== Книги 41-80 по длине ===
  41. Picture_of_Dorian_Gray                             |  437,768 симв. |  241 имён
  42. Girl_Online                                        |  439,846 симв. |  145 имён
  43. Northanger_Abbey                                   |  441,994 симв. |   88 имён
  44. Royal_Fashion_2_All_the_Gossip_From_Paris          |  460,052 симв. |  101 имён
  45. The_Quiet_Mother                                   |  472,055 симв. |  102 имён
  46. Persuasion                                         |  473,116 симв. |  108 имён
  47. Sands_of_the_Soul                                  |  483,732 симв. |   82 имён
  48. This_Side_of_Paradise                              |  484,612 симв. |  345 имён
  49. The_Scarlet_Letter                                 |  488,436 симв. |   87 имён
  50. The_Hobbit                                         |  507,429 симв. |   90 имён
  51. Ice_Station_Zebra                                  |  528,204 симв. |   95 имён
  52. The_Secret_Agent   

In [67]:
# === Следующие 40 книг (с 41 по 80 по длине) ===

from pathlib import Path

# 1. Собираем статистику по книгам и сортируем по длине
book_lengths = []
for book_name, text in book_name_to_text_final.items():
    names_count = len(combined_df[combined_df["book_name"] == book_name])
    book_lengths.append((book_name, len(text), names_count))

book_lengths.sort(key=lambda x: x[1])  # по возрастанию длины

# 2. Берём следующие 40 книг (с 40 по 80)
START_IDX = 80
END_IDX = 100
books_to_test = book_lengths[START_IDX:END_IDX]

print(f"=== Книги {START_IDX+1}-{min(END_IDX, len(book_lengths))} по длине ===")
for i, (bn, length, names_count) in enumerate(books_to_test, START_IDX + 1):
    print(f"  {i:2d}. {bn[:50]:<50} | {length:>8,} симв. | {names_count:>4} имён")

print()

# 3. Создаём новый датасет для результатов
all_results = []

# 4. Цикл по книгам
for i, (book_name, text_len, names_count) in enumerate(books_to_test, START_IDX + 1):
    print(f"\n[{i}/{END_IDX}] === {book_name} ===")
    print(f"  Длина текста: {text_len:,} символов")
    print(f"  Имен в книге: {names_count}")

    text = book_name_to_text_final[book_name]
    names_for_book = combined_df[combined_df["book_name"] == book_name].copy()

    # Обработка
    print(f"  [INFO] Обработка...")
    try:
        result = process_book(text, names_for_book)
    except Exception as e:
        print(f"  [FAIL] {e}")
        continue

    # Собираем результаты
    book_results = []
    for idx, row in names_for_book.iterrows():
        n = normalize_for_match(row["name"])
        if n in result:
            adj_dict = result[n]["adjectives"]
            total = result[n]["total_adjectives"]
            if total > 0:
                top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
                top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
                book_results.append({
                    "book_name": book_name,
                    "name": row["name"],
                    "llm_classification": row["llm_classification"],
                    "total_adjectives": total,
                    "top_adjectives": top_str,
                    "adjectives": adj_dict,
                })

    # Добавляем в общий список
    all_results.extend(book_results)

    # Отладка: пример для этой книги
    if book_results:
        top_example = max(book_results, key=lambda x: x["total_adjectives"])
        print(f"  [OK] пример: '{top_example['name']}' | total={top_example['total_adjectives']} | top: {top_example['top_adjectives']}")
    else:
        print(f"  [OK] нет прилагательных")
    print(f"  Найдено имён с прилагательными: {len(book_results)}")

# 5. Формируем DataFrame новых результатов
new_results_df = pd.DataFrame(all_results)

print(f"\n{'='*60}")
print(f"=== ИТОГ для книг {START_IDX+1}-{END_IDX} ===")
print(f"Всего новых записей: {len(new_results_df)}")
if not new_results_df.empty:
    print(f"Книг с данными: {new_results_df['book_name'].nunique()}")
    print(f"Суммарное количество прилагательных: {new_results_df['total_adjectives'].sum():,}")

# 6. === ДОПИСЫВАЕМ В СУЩЕСТВУЮЩИЙ ФАЙЛ ===
OUTPUT_DIR = Path("/Users/kseniazavyalova/Desktop/ner_results_batch")
OUTPUT_TEST_CSV = OUTPUT_DIR / "test_short_books_adjectives.csv"
OUTPUT_TEST_JSON = OUTPUT_DIR / "test_short_books_adjectives.json"

if not new_results_df.empty:
    # Читаем существующий CSV
    if OUTPUT_TEST_CSV.exists():
        existing_df = pd.read_csv(OUTPUT_TEST_CSV)
        print(f"\n📖 Прочитан существующий файл: {len(existing_df)} строк")
        
        # Конкатенируем
        combined_results_df = pd.concat([existing_df, new_results_df], ignore_index=True)
    else:
        print(f"\n[WARN] Файл не найден, создаём новый")
        combined_results_df = new_results_df
    
    # Сохраняем CSV
    combined_results_df.to_csv(OUTPUT_TEST_CSV, index=False)
    print(f"\n💾 Сохранено (CSV):  {OUTPUT_TEST_CSV}")
    print(f"   Всего строк: {len(combined_results_df)}")
    print(f"   Размер: {OUTPUT_TEST_CSV.stat().st_size / 1024:.1f} KB")

    # JSON
    combined_results_df.to_json(OUTPUT_TEST_JSON, orient="records", force_ascii=False, indent=2)
    print(f"💾 Сохранено (JSON): {OUTPUT_TEST_JSON}")
    print(f"   Размер: {OUTPUT_TEST_JSON.stat().st_size / 1024:.1f} KB")
else:
    print("\n[WARN] Нет новых данных для сохранения")

=== Книги 81-100 по длине ===
  81. Far_from_the_Madding_Crowd                         |  790,196 симв. |  221 имён
  82. Gideon_the_Ninth                                   |  810,634 симв. |  170 имён
  83. City_of_Glass                                      |  839,589 симв. |  218 имён
  84. Tess_of_the_dUrbervilles                           |  858,308 симв. |  210 имён
  85. Dracula                                            |  861,148 симв. |  184 имён
  86. Portrait_of_a_Lady                                 |  873,507 симв. | 1604 имён
  87. Emma                                               |  880,423 симв. |  136 имён
  88. Evelina                                            |  887,681 симв. |  147 имён
  89. Oliver_Twist                                       |  893,020 симв. |  126 имён
  90. CHAPTER_ONE                                        |  918,285 симв. |  183 имён
  91. The_Tenant_of_Wildfell_Hall                        |  953,587 симв. |  136 имён
  92. North_and_South   

In [68]:
# === Продолжить обработку с уже обработанных книг до конца ===

from pathlib import Path
from tqdm import tqdm

OUTPUT_DIR = Path("/Users/kseniazavyalova/Desktop/ner_results_batch")
OUTPUT_TEST_CSV = OUTPUT_DIR / "test_short_books_adjectives.csv"
OUTPUT_TEST_JSON = OUTPUT_DIR / "test_short_books_adjectives.json"

# 1. Читаем существующий файл
if OUTPUT_TEST_CSV.exists():
    existing_df = pd.read_csv(OUTPUT_TEST_CSV)
    already_done = set(existing_df["book_name"].unique())
    print(f"📖 Прочитан существующий файл: {len(existing_df)} строк")
    print(f"   Уже обработано книг: {len(already_done)}")
else:
    existing_df = pd.DataFrame()
    already_done = set()
    print("[INFO] Файл не найден — начинаем с нуля")

# 2. Собираем все книги и исключаем уже обработанные
book_lengths = []
for book_name, text in book_name_to_text_final.items():
    if book_name in already_done:
        continue
    names_count = len(combined_df[combined_df["book_name"] == book_name])
    book_lengths.append((book_name, len(text), names_count))

# Сортируем по возрастанию длины — короткие обрабатываются быстрее
book_lengths.sort(key=lambda x: x[1])

print(f"Осталось обработать книг: {len(book_lengths)}")
print(f"\n=== Книги к обработке ===")
for i, (bn, length, names_count) in enumerate(book_lengths, 1):
    print(f"  {i:2d}. {bn[:50]:<50} | {length:>8,} симв. | {names_count:>4} имён")
print()

# 3. Цикл обработки с прогресс-баром
all_results = []
processed = 0
skipped = 0

pbar = tqdm(book_lengths, desc="Обработка книг", unit="book")

for book_name, text_len, names_count in pbar:
    pbar.set_postfix({
        "book": book_name[:30],
        "names": names_count,
        "chars": f"{text_len//1000}k",
    })

    text = book_name_to_text_final[book_name]
    names_for_book = combined_df[combined_df["book_name"] == book_name].copy()

    try:
        result = process_book(text, names_for_book)
    except Exception as e:
        pbar.write(f"  [FAIL] {book_name}: {e}")
        skipped += 1
        continue

    book_results = []
    for idx, row in names_for_book.iterrows():
        n = normalize_for_match(row["name"])
        if n in result:
            adj_dict = result[n]["adjectives"]
            total = result[n]["total_adjectives"]
            if total > 0:
                top_adjs = sorted(adj_dict.items(), key=lambda x: x[1], reverse=True)[:5]
                top_str = ", ".join(f"{a}:{c}" for a, c in top_adjs)
                book_results.append({
                    "book_name": book_name,
                    "name": row["name"],
                    "llm_classification": row["llm_classification"],
                    "total_adjectives": total,
                    "top_adjectives": top_str,
                    "adjectives": adj_dict,
                })

    all_results.extend(book_results)
    processed += 1

    # Отладка
    if book_results:
        top_example = max(book_results, key=lambda x: x["total_adjectives"])
        pbar.write(f"  [OK] пример: '{top_example['name']}' | total={top_example['total_adjectives']} | top: {top_example['top_adjectives']}")
    else:
        pbar.write(f"  [OK] {book_name[:40]} | нет прилагательных")

    # Промежуточное сохранение каждые 5 книг
    if processed % 5 == 0 and all_results:
        new_df = pd.DataFrame(all_results)
        if existing_df.empty:
            combined_results_df = new_df
        else:
            combined_results_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_results_df.to_csv(OUTPUT_TEST_CSV, index=False)
        pbar.write(f"  💾 Промежуточное сохранение ({processed} книг)")

pbar.close()

# 4. Финальное сохранение
new_results_df = pd.DataFrame(all_results)

print(f"\n{'='*60}")
print(f"=== ИТОГ ===")
print(f"Обработано книг: {processed}")
print(f"Пропущено с ошибкой: {skipped}")
print(f"Новых записей: {len(new_results_df)}")

if not new_results_df.empty:
    if existing_df.empty:
        combined_results_df = new_results_df
    else:
        combined_results_df = pd.concat([existing_df, new_results_df], ignore_index=True)

    # CSV
    combined_results_df.to_csv(OUTPUT_TEST_CSV, index=False)
    print(f"\n💾 Сохранено (CSV):  {OUTPUT_TEST_CSV}")
    print(f"   Всего строк: {len(combined_results_df)}")
    print(f"   Размер: {OUTPUT_TEST_CSV.stat().st_size / 1024:.1f} KB")

    # JSON
    combined_results_df.to_json(OUTPUT_TEST_JSON, orient="records", force_ascii=False, indent=2)
    print(f"💾 Сохранено (JSON): {OUTPUT_TEST_JSON}")
    print(f"   Размер: {OUTPUT_TEST_JSON.stat().st_size / 1024:.1f} KB")

    print(f"\nКниг обработано всего: {combined_results_df['book_name'].nunique()}")
else:
    print("\n[WARN] Нет новых данных для сохранения")

📖 Прочитан существующий файл: 2511 строк
   Уже обработано книг: 98
Осталось обработать книг: 10

=== Книги к обработке ===
   1. Shirley                                            |   21,238 симв. |    6 имён
   2. The_Castle_of_Otranto                              |   50,731 симв. |    7 имён
   3. CRESCENT_CITY_HOUSE_OF_FLAME_AND_SHADOW            | 1,409,127 симв. |  376 имён
   4. The_Mysteries_of_Udolpho                           | 1,710,352 симв. |  222 имён
   5. A_Feast_for_Crows                                  | 1,713,576 симв. | 1740 имён
   6. Vanity_Fair                                        | 1,745,856 симв. |  804 имён
   7. Middlemarch                                        | 1,780,107 симв. |  504 имён
   8. Tom_Jones                                          | 1,960,956 симв. |  368 имён
   9. David_Copperfield                                  | 1,972,851 симв. |  285 имён
  10. Bleak_House                                        | 1,978,971 симв. |  278 имён



Обработка книг:  10%|█         | 1/10 [00:01<00:12,  1.41s/book, book=The_Castle_of_Otranto, names=7, chars=50k]

  [OK] Shirley | нет прилагательных


Обработка книг:  20%|██        | 2/10 [00:03<00:14,  1.83s/book, book=CRESCENT_CITY_HOUSE_OF_FLAME_A, names=376, chars=1409k]

  [OK] The_Castle_of_Otranto | нет прилагательных


...

  [OK] пример: 'sophia' | total=82 | top: dear:11, lovely:9, charming:4, pleased:3, satisfied:3


Обработка книг:  90%|█████████ | 9/10 [1:20:10<09:44, 584.92s/book, book=Bleak_House, names=278, chars=1978k]      

  [OK] пример: 'he' | total=520 | top: glad:16, good:12, old:10, able:10, ready:9


Обработка книг: 100%|██████████| 10/10 [1:26:58<00:00, 521.85s/book, book=Bleak_House, names=278, chars=1978k]

  [OK] пример: 'he' | total=758 | top: good:26, dead:20, ill:19, fond:18, old:15
  💾 Промежуточное сохранение (10 книг)

=== ИТОГ ===
Обработано книг: 10
Пропущено с ошибкой: 0
Новых записей: 774

💾 Сохранено (CSV):  /Users/kseniazavyalova/Desktop/ner_results_batch/test_short_books_adjectives.csv
   Всего строк: 3285
   Размер: 402.8 KB
💾 Сохранено (JSON): /Users/kseniazavyalova/Desktop/ner_results_batch/test_short_books_adjectives.json
   Размер: 852.7 KB

Книг обработано всего: 106


In [69]:
# === Создание датафрейма метаданных только для обработанных книг ===

import pandas as pd
from pathlib import Path
import re

# === 1. Загружаем список обработанных книг ===
print("[INFO] Загрузка списка обработанных книг...")
PROCESSED_CSV = Path("/Users/kseniazavyalova/Desktop/ner_results_batch/test_short_books_adjectives.csv")
processed_df = pd.read_csv(PROCESSED_CSV)
processed_books = set(processed_df["book_name"].unique())
print(f"Обработано книг: {len(processed_books)}")

# === 2. Загружаем метаданные из исходных CSV ===
books_df_1 = pd.read_csv("/Users/kseniazavyalova/Desktop/all_books_data.csv")
books_df_2 = pd.read_csv("/Users/kseniazavyalova/Desktop/all_books_data2.csv")
books_meta_df = pd.concat([books_df_1, books_df_2], ignore_index=True)

# === 3. Функция нормализации ===
def normalize_title_to_book_name(title: str) -> str:
    s = str(title)
    s = re.sub(r"[^\w\s]", "_", s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

# === 4. CORPUS с метаданными ===
CORPUS = {
    # Женщины-авторы
    "Evelina": {"author": "Burney_Fanny", "gender": "female", "genre": "satire", "year": 1778},
    "The_Mysteries_of_Udolpho": {"author": "Radcliffe_Ann", "gender": "female", "genre": "gothic", "year": 1794},
    "Mary_A_Fiction": {"author": "Wollstonecraft_Mary", "gender": "female", "genre": "philosophical", "year": 1788},
    "Charlotte_Temple": {"author": "Rowson_Susanna", "gender": "female", "genre": "sentimental", "year": 1791},
    "Sense_and_Sensibility": {"author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1811},
    "Pride_and_Prejudice": {"author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1813},
    "Emma": {"author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1815},
    "Persuasion": {"author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1818},
    "Frankenstein": {"author": "Shelley_Mary", "gender": "female", "genre": "gothic_scifi", "year": 1818},
    "The_Last_Man": {"author": "Shelley_Mary", "gender": "female", "genre": "scifi", "year": 1826},
    "Northanger_Abbey": {"author": "Austen_Jane", "gender": "female", "genre": "romance", "year": 1817},
    "The_Tenant_of_Wildfell_Hall": {"author": "Bronte_Anne", "gender": "female", "genre": "realism", "year": 1848},
    "Jane_Eyre": {"author": "Bronte_Charlotte", "gender": "female", "genre": "gothic", "year": 1847},
    "Villette": {"author": "Bronte_Charlotte", "gender": "female", "genre": "psychological", "year": 1853},
    "Shirley": {"author": "Bronte_Charlotte", "gender": "female", "genre": "social", "year": 1849},
    "Wuthering_Heights": {"author": "Bronte_Emily", "gender": "female", "genre": "gothic", "year": 1847},
    "Agnes_Grey": {"author": "Bronte_Anne", "gender": "female", "genre": "realism", "year": 1847},
    "Middlemarch": {"author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1871},
    "Silas_Marner": {"author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1861},
    "The_Mill_on_the_Floss": {"author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1860},
    "Adam_Bede": {"author": "Eliot_George", "gender": "female", "genre": "realism", "year": 1859},
    "North_and_South": {"author": "Gaskell_Elizabeth", "gender": "female", "genre": "social", "year": 1855},
    "Cranford": {"author": "Gaskell_Elizabeth", "gender": "female", "genre": "comedy", "year": 1853},
    "Mary_Barton": {"author": "Gaskell_Elizabeth", "gender": "female", "genre": "social", "year": 1848},
    "The_Awakening": {"author": "Chopin_Kate", "gender": "female", "genre": "psychological", "year": 1899},
    "The_Story_of_an_Hour": {"author": "Chopin_Kate", "gender": "female", "genre": "short_story", "year": 1894},
    "The_Country_of_the_Pointed_Firs": {"author": "Jewett_Sarah", "gender": "female", "genre": "regionalism", "year": 1896},
    "A_Country_Doctor": {"author": "Jewett_Sarah", "gender": "female", "genre": "regionalism", "year": 1884},
    "Herland": {"author": "Gilman_Charlotte", "gender": "female", "genre": "utopia", "year": 1915},
    "The_Yellow_Wallpaper": {"author": "Gilman_Charlotte", "gender": "female", "genre": "psychological", "year": 1892},
    "The_Age_of_Innocence": {"author": "Wharton_Edith", "gender": "female", "genre": "social", "year": 1920},
    "The_House_of_Mirth": {"author": "Wharton_Edith", "gender": "female", "genre": "social", "year": 1905},
    "Ethan_Frome": {"author": "Wharton_Edith", "gender": "female", "genre": "tragedy", "year": 1911},
    "Little_Women": {"author": "Alcott_Louisa", "gender": "female", "genre": "family", "year": 1868},
    "Little_Men": {"author": "Alcott_Louisa", "gender": "female", "genre": "family", "year": 1871},
    "Jos_Boys": {"author": "Alcott_Louisa", "gender": "female", "genre": "family", "year": 1886},
    "Anne_of_Green_Gables": {"author": "Montgomery_LM", "gender": "female", "genre": "coming_of_age", "year": 1908},
    "Emily_of_New_Moon": {"author": "Montgomery_LM", "gender": "female", "genre": "coming_of_age", "year": 1923},
    "The_Secret_Garden": {"author": "Burnett_Frances", "gender": "female", "genre": "children", "year": 1911},
    "A_Little_Princess": {"author": "Burnett_Frances", "gender": "female", "genre": "children", "year": 1905},
    
    # Мужчины-авторы
    "Tom_Jones": {"author": "Fielding_Henry", "gender": "male", "genre": "picaresque", "year": 1749},
    "Gullivers_Travels": {"author": "Swift_Jonathan", "gender": "male", "genre": "satire", "year": 1726},
    "Robinson_Crusoe": {"author": "Defoe_Daniel", "gender": "male", "genre": "adventure", "year": 1719},
    "Tristram_Shandy": {"author": "Sterne_Laurence", "gender": "male", "genre": "experimental", "year": 1759},
    "The_Castle_of_Otranto": {"author": "Walpole_Horace", "gender": "male", "genre": "gothic", "year": 1764},
    "The_Monk": {"author": "Lewis_Matthew", "gender": "male", "genre": "gothic", "year": 1796},
    "The_Scarlet_Letter": {"author": "Hawthorne_Nathaniel", "gender": "male", "genre": "historical", "year": 1850},
    "House_of_Seven_Gables": {"author": "Hawthorne_Nathaniel", "gender": "male", "genre": "gothic", "year": 1851},
    "Moby_Dick": {"author": "Melville_Herman", "gender": "male", "genre": "adventure", "year": 1851},
    "Bartleby_the_Scrivener": {"author": "Melville_Herman", "gender": "male", "genre": "psychological", "year": 1853},
    "David_Copperfield": {"author": "Dickens_Charles", "gender": "male", "genre": "realism", "year": 1849},
    "Great_Expectations": {"author": "Dickens_Charles", "gender": "male", "genre": "realism", "year": 1861},
    "Oliver_Twist": {"author": "Dickens_Charles", "gender": "male", "genre": "social", "year": 1838},
    "Bleak_House": {"author": "Dickens_Charles", "gender": "male", "genre": "social", "year": 1853},
    "Vanity_Fair": {"author": "Thackeray_William", "gender": "male", "genre": "satire", "year": 1848},
    "Tom_Sawyer": {"author": "Twain_Mark", "gender": "male", "genre": "adventure", "year": 1876},
    "Huckleberry_Finn": {"author": "Twain_Mark", "gender": "male", "genre": "adventure", "year": 1884},
    "Prince_and_the_Pauper": {"author": "Twain_Mark", "gender": "male", "genre": "historical", "year": 1881},
    "Tess_of_the_dUrbervilles": {"author": "Hardy_Thomas", "gender": "male", "genre": "naturalism", "year": 1891},
    "Far_from_the_Madding_Crowd": {"author": "Hardy_Thomas", "gender": "male", "genre": "realism", "year": 1874},
    "Jude_the_Obscure": {"author": "Hardy_Thomas", "gender": "male", "genre": "tragedy", "year": 1895},
    "Picture_of_Dorian_Gray": {"author": "Wilde_Oscar", "gender": "male", "genre": "gothic", "year": 1890},
    "Importance_of_Being_Earnest": {"author": "Wilde_Oscar", "gender": "male", "genre": "comedy", "year": 1895},
    "Dracula": {"author": "Stoker_Bram", "gender": "male", "genre": "gothic", "year": 1897},
    "War_of_the_Worlds": {"author": "Wells_HG", "gender": "male", "genre": "scifi", "year": 1897},
    "The_Invisible_Man": {"author": "Wells_HG", "gender": "male", "genre": "scifi", "year": 1897},
    "The_Time_Machine": {"author": "Wells_HG", "gender": "male", "genre": "scifi", "year": 1895},
    "Heart_of_Darkness": {"author": "Conrad_Joseph", "gender": "male", "genre": "psychological", "year": 1899},
    "Lord_Jim": {"author": "Conrad_Joseph", "gender": "male", "genre": "adventure", "year": 1900},
    "The_Secret_Agent": {"author": "Conrad_Joseph", "gender": "male", "genre": "political", "year": 1907},
    "Portrait_of_a_Lady": {"author": "James_Henry", "gender": "male", "genre": "psychological", "year": 1881},
    "Turn_of_the_Screw": {"author": "James_Henry", "gender": "male", "genre": "gothic", "year": 1898},
    "Daisy_Miller": {"author": "James_Henry", "gender": "male", "genre": "psychological", "year": 1878},
    "Adventures_of_Sherlock_Holmes": {"author": "Doyle_Arthur", "gender": "male", "genre": "detective", "year": 1892},
    "Hound_of_the_Baskervilles": {"author": "Doyle_Arthur", "gender": "male", "genre": "detective", "year": 1902},
    "Sister_Carrie": {"author": "Dreiser_Theodore", "gender": "male", "genre": "naturalism", "year": 1900},
    "An_American_Tragedy": {"author": "Dreiser_Theodore", "gender": "male", "genre": "naturalism", "year": 1925},
    "This_Side_of_Paradise": {"author": "Fitzgerald_FScott", "gender": "male", "genre": "modernist", "year": 1920},
    "The_Beautiful_and_Damned": {"author": "Fitzgerald_FScott", "gender": "male", "genre": "modernist", "year": 1922},
}

# === 5. Собираем метаданные только для обработанных книг ===
books_metadata = []

for book_name in sorted(processed_books):
    # Длина текста
    text_length = len(book_name_to_text_final.get(book_name, ""))
    
    # Подсчёт персонажей по полу из combined_df по llm_classification
    book_names_df = combined_df[combined_df["book_name"] == book_name]
    
    # llm_classification: 1 = female, 2 = male
    female_count = len(book_names_df[book_names_df["llm_classification"].astype(str) == "1"])
    male_count = len(book_names_df[book_names_df["llm_classification"].astype(str) == "2"])
    total_names = len(book_names_df)
    
    # Соотношение
    gender_ratio = female_count / male_count if male_count > 0 else (float('inf') if female_count > 0 else 0)
    
    # Плотность имён (на 100k символов)
    names_density = (total_names / text_length * 100_000) if text_length > 0 else 0
    
    # Метаданные из CORPUS или books_meta_df
    author = None
    author_gender = None
    genre = None
    year = None
    century = None
    
    if book_name in CORPUS:
        meta = CORPUS[book_name]
        author = meta["author"]
        author_gender = meta["gender"]
        genre = meta["genre"]
        year = meta["year"]
        century = (year - 1) // 100 + 1
    else:
        # Ищем в books_meta_df
        matching = books_meta_df[books_meta_df["Title"].apply(normalize_title_to_book_name) == book_name]
        if not matching.empty:
            book_row = matching.iloc[0]
            author = book_row.get("Authors", "")
            genre = book_row.get("Genre", "")
    
    books_metadata.append({
        "book_name": book_name,
        "text_length": text_length,
        "year": year,
        "century": century,
        "author": author,
        "author_gender": author_gender,
        "genre": genre,
        "total_names": total_names,
        "female_names_count": female_count,
        "male_names_count": male_count,
        "gender_ratio": gender_ratio,
        "names_density_per_100k": round(names_density, 2),
    })

books_metadata_df = pd.DataFrame(books_metadata)

# Сортируем по году (если есть), потом по названию
books_metadata_df = books_metadata_df.sort_values(
    ["year", "book_name"], 
    na_position="last"
).reset_index(drop=True)

# === 6. Вывод ===
print(f"\n[INFO] Итоговый датафрейм: {len(books_metadata_df)} книг")
display(books_metadata_df)

print(f"\n=== Статистика ===")
print(f"Книг с известным годом: {books_metadata_df['year'].notna().sum()}")
print(f"Книг с известным автором: {books_metadata_df['author'].notna().sum()}")
print(f"Книг с известным полом автора: {books_metadata_df['author_gender'].notna().sum()}")
print(f"\nРаспределение по полу автора:")
print(books_metadata_df['author_gender'].value_counts(dropna=False))
print(f"\nРаспределение по жанрам:")
print(books_metadata_df['genre'].value_counts(dropna=False))
print(f"\nРаспределение по векам:")
print(books_metadata_df['century'].value_counts(dropna=False).sort_index())


[INFO] Загрузка списка обработанных книг...
Обработано книг: 106

[INFO] Итоговый датафрейм: 106 книг


,book_name,text_length,year,century,author,author_gender,genre,total_names,female_names_count,male_names_count,gender_ratio,names_density_per_100k
0,Robinson_Crusoe,631685,1719.0,18.0,Defoe_Daniel,male,adventure,48,13,35,0.371429,7.60
1,Gullivers_Travels,581556,1726.0,18.0,Swift_Jonathan,male,satire,110,20,90,0.222222,18.91
2,Tom_Jones,1960956,1749.0,18.0,Fielding_Henry,male,picaresque,368,122,246,0.495935,18.77
3,Tristram_Shandy,1057936,1759.0,18.0,Sterne_Laurence,male,experimental,806,214,592,0.361486,76.19
4,Evelina,887681,1778.0,18.0,Burney_Fanny,female,satire,147,71,76,0.934211,16.56
...,...,...,...,...,...,...,...,...,...,...,...,...
101,Twilight,657972,NaN,NaN,"Stephenie Meyer, Your Name",None,love_detective,111,50,61,0.819672,16.87
102,Veronika_decides_to_die,289512,NaN,NaN,"Paulo Coelho, Paulo Coelho",None,"prose_contemporary, prose_contemporary",52,24,28,0.857143,17.96
103,Warlock_s_Last_Ride,553058,NaN,NaN,Christopher Stasheff,None,sf_fantasy,175,65,110,0.590909,31.64
104,Whimbrel_House_1_Keeper_of_Enchanted_Rooms,553034,NaN,NaN,"Holmberg, Charlie N., Holmberg, Charlie N.",None,antique,121,59,62,0.951613,21.88



=== Статистика ===
Книг с известным годом: 73
Книг с известным автором: 106
Книг с известным полом автора: 73

Распределение по полу автора:
author_gender
female    37
male      36
None      33
Name: count, dtype: int64

Распределение по жанрам:
genre
sf_fantasy                                       11
antique                                          10
realism                                           9
gothic                                            8
psychological                                     7
social                                            6
adventure                                         5
romance                                           5
scifi                                             4
satire                                            3
family                                            2
modernist                                         2
coming_of_age                                     2
detective                                         2
regionalism        

In [70]:
books_metadata_df

,book_name,text_length,year,century,author,author_gender,genre,total_names,female_names_count,male_names_count,gender_ratio,names_density_per_100k
0,Robinson_Crusoe,631685,1719.0,18.0,Defoe_Daniel,male,adventure,48,13,35,0.371429,7.60
1,Gullivers_Travels,581556,1726.0,18.0,Swift_Jonathan,male,satire,110,20,90,0.222222,18.91
2,Tom_Jones,1960956,1749.0,18.0,Fielding_Henry,male,picaresque,368,122,246,0.495935,18.77
3,Tristram_Shandy,1057936,1759.0,18.0,Sterne_Laurence,male,experimental,806,214,592,0.361486,76.19
4,Evelina,887681,1778.0,18.0,Burney_Fanny,female,satire,147,71,76,0.934211,16.56
...,...,...,...,...,...,...,...,...,...,...,...,...
101,Twilight,657972,NaN,NaN,"Stephenie Meyer, Your Name",None,love_detective,111,50,61,0.819672,16.87
102,Veronika_decides_to_die,289512,NaN,NaN,"Paulo Coelho, Paulo Coelho",None,"prose_contemporary, prose_contemporary",52,24,28,0.857143,17.96
103,Warlock_s_Last_Ride,553058,NaN,NaN,Christopher Stasheff,None,sf_fantasy,175,65,110,0.590909,31.64
104,Whimbrel_House_1_Keeper_of_Enchanted_Rooms,553034,NaN,NaN,"Holmberg, Charlie N., Holmberg, Charlie N.",None,antique,121,59,62,0.951613,21.88


In [71]:
OUTPUT_METADATA_CSV = Path("/Users/kseniazavyalova/Desktop/ner_results_batch/books_metadata.csv")
books_metadata_df.to_csv(OUTPUT_METADATA_CSV, index=False)
print(f"\n💾 Сохранено: {OUTPUT_METADATA_CSV}")


💾 Сохранено: /Users/kseniazavyalova/Desktop/ner_results_batch/books_metadata.csv


In [74]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("/Users/kseniazavyalova/Desktop/meta_stats_of_book.csv")
df = pd.read_csv(CSV_PATH)

# === Финальный маппинг ===
FINAL_GENRE_UNION = {
    # Готика
    "gothic": "gothic",
    "gothic_scifi": "gothic",
    
    # Романтизм
    "romance": "romance",
    "sentimental": "romance",
    "love_contemporary": "romance",  # ← добавлено
    
    # Реализм
    "realism": "realism",
    "naturalism": "realism",
    
    # Психологическая проза
    "psychological": "psychological",
    "psychological_thriller": "psychological",
    
    # Сатира и комедия
    "satire": "satire",
    "comedy": "satire",
    "picaresque": "satire",
    
    # Социальная проза
    "social": "social",
    "political": "social",
    
    # Историческая проза
    "historical": "historical",
    
    # Философская и экспериментальная
    "philosophical": "philosophical",
    "experimental": "experimental",
    "modernist": "modernist",
    
    # Трагедия
    "tragedy": "tragedy",
    
    # Короткая форма
    "short_story": "short_story",
    
    # Регионализм
    "regionalism": "regionalism",
    
    # Утопия
    "utopia": "utopia",
    
    # Фантастика и фэнтези
    "scifi": "scifi",
    "sf": "scifi",
    "sf_epic": "scifi",
    "fantasy": "fantasy",
    "sf_fantasy": "fantasy",
    "sf_fantasy_city": "fantasy",
    "urban_fantasy": "fantasy",
    
    # Детская литература
    "children": "children",
    "child_adv": "children",
    "child_tale": "children",
    "children_adventure": "children",
    "children_fairy_tale": "children",
    "family": "children",
    "coming_of_age": "coming_of_age",
    
    # Детективы и триллеры
    "detective": "detective",
    "crime": "detective",
    "det_crime": "detective",
    "crime_thriller": "detective",
    "love_detective": "detective",  # ← добавлено
    "thriller": "thriller",
    "action_thriller": "thriller",
    "thriller, det_action": "thriller",  # ← добавлено
    "romantic_suspense": "thriller",
    
    # Современная проза
    "contemporary_fiction": "contemporary",
    "contemporary_romance": "contemporary",
    "prose_contemporary": "contemporary",
    "prose_contemporary, prose_contemporary": "contemporary",
    
    # Биографии и мемуары
    "biography": "biography",
    "memoir": "biography",
    "nonf_biography": "biography",
    "nonf_biography, home_pets, prose_contemporary": "biography",
    
    # Приключения
    "adventure": "adventure",
    
    # antique → будет заменён отдельно после просмотра списка
}

def normalize_genre(g):
    if pd.isna(g) or not str(g).strip():
        return "unknown"
    g = str(g).strip().lower()
    if "," in g:
        g = g.split(",")[0].strip()
    return FINAL_GENRE_UNION.get(g, g)

# Применяем маппинг
df["genre"] = df["genre"].apply(normalize_genre)

# Показываем результат
print(f"\n=== Жанры ПОСЛЕ финального объединения ({df['genre'].nunique()} уникальных) ===")
print(df['genre'].value_counts(dropna=False).sort_index())

# Проверяем, что осталось необъединённым
unmapped = df[~df['genre'].isin(FINAL_GENRE_UNION.values())]['genre'].unique()
if len(unmapped) > 0:
    print(f"\n[WARN] Остались необъединённые жанры: {unmapped}")

# Сохраняем
df.to_csv(CSV_PATH, index=False)
print(f"\n💾 Сохранено: {CSV_PATH}")


=== Жанры ПОСЛЕ финального объединения (24 уникальных) ===
genre
adventure         5
antique          10
biography         1
children          5
coming_of_age     2
contemporary      2
detective         4
experimental      1
fantasy          12
gothic            9
historical        2
modernist         2
philosophical     1
psychological     7
realism          10
regionalism       2
romance           7
satire            6
scifi             6
short_story       1
social            7
thriller          1
tragedy           2
utopia            1
Name: count, dtype: int64

[WARN] Остались необъединённые жанры: ['antique']

💾 Сохранено: /Users/kseniazavyalova/Desktop/meta_stats_of_book.csv


In [75]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("/Users/kseniazavyalova/Desktop/meta_stats_of_book.csv")
df = pd.read_csv(CSV_PATH)

# === Ручной маппинг для книг с жанром 'antique' ===
ANTIQUE_FIX = {
    "BRIDE": "romance",                                    # Ali Hazelwood — романтическая комедия
    "CHAPTER_ONE": "contemporary",                         # harry, harry — современная проза
    "CRESCENT_CITY_HOUSE_OF_FLAME_AND_SHADOW": "fantasy",  # Sarah J. Maas — urban fantasy
    "Juniper_Thorn": "fantasy",                            # Ava Reid — dark fantasy
    "Roots_of_Chaos_3_Among_the_Burning_Flowers": "fantasy",  # Samantha Shannon — epic fantasy
    "Royal_Fashion_2_All_the_Gossip_From_Paris": "nonfiction",  # Jessica Gregory — non-fiction
    "Tamora_Pierce_The_Will_of_the": "fantasy",            # Tamora Pierce — фэнтези
    "The_Will_of_the_Many": "fantasy",                     # James Islington — epic fantasy
    "Whimbrel_House_1_Keeper_of_Enchanted_Rooms": "fantasy",  # Charlie N. Holmberg — fantasy
    "Zodiac_Academy_4_Shadow_Princess_An_Academy_Bully_Romance_Supernatural_Bullies_and_Beasts": "fantasy",  # reverse harem fantasy
}

# Заменяем antique на правильные жанры
for book_name, correct_genre in ANTIQUE_FIX.items():
    mask = (df["book_name"] == book_name) & (df["genre"] == "antique")
    df.loc[mask, "genre"] = correct_genre

# Проверяем, что antique больше нет
antique_remaining = df[df["genre"] == "antique"]
if len(antique_remaining) > 0:
    print(f"[WARN] Осталось книг с жанром 'antique': {len(antique_remaining)}")
    print(antique_remaining[["book_name", "genre"]])
else:
    print("[OK] Все 'antique' заменены")

# Показываем финальное распределение
print(f"\n=== Финальное распределение жанров ({df['genre'].nunique()} уникальных) ===")
print(df['genre'].value_counts(dropna=False).sort_index())

# Сохраняем
df.to_csv(CSV_PATH, index=False)
print(f"\n💾 Сохранено: {CSV_PATH}")

[OK] Все 'antique' заменены

=== Финальное распределение жанров (24 уникальных) ===
genre
adventure         5
biography         1
children          5
coming_of_age     2
contemporary      3
detective         4
experimental      1
fantasy          19
gothic            9
historical        2
modernist         2
nonfiction        1
philosophical     1
psychological     7
realism          10
regionalism       2
romance           8
satire            6
scifi             6
short_story       1
social            7
thriller          1
tragedy           2
utopia            1
Name: count, dtype: int64

💾 Сохранено: /Users/kseniazavyalova/Desktop/meta_stats_of_book.csv


In [76]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("/Users/kseniazavyalova/Desktop/meta_stats_of_book.csv")
df = pd.read_csv(CSV_PATH)

# === Дополнительное объединение ===
ADDITIONAL_UNION = {
    # Психологическая проза
    "psychological": "psychological",
    "experimental": "psychological",      # экспериментальная проза часто психологическая
    "philosophical": "psychological",     # философская проза часто психологическая
    
    # Детективы и триллеры
    "detective": "mystery_thriller",
    "thriller": "mystery_thriller",
    
    # Nonfiction
    "biography": "nonfiction",
    "nonfiction": "nonfiction",
    
    # Реализм (включая регионализм)
    "realism": "realism",
    "regionalism": "realism",             # регионализм — подвид реализма
    
    # Остальное оставляем как есть
    "adventure": "adventure",
    "children": "children",
    "coming_of_age": "coming_of_age",
    "contemporary": "contemporary",
    "fantasy": "fantasy",
    "gothic": "gothic",
    "historical": "historical",
    "modernist": "modernist",
    "romance": "romance",
    "satire": "satire",
    "scifi": "scifi",
    "short_story": "short_story",
    "social": "social",
    "tragedy": "tragedy",
    "utopia": "utopia",
}

df["genre"] = df["genre"].apply(lambda g: ADDITIONAL_UNION.get(g, g))

print(f"=== После дополнительного объединения ({df['genre'].nunique()} уникальных) ===")
print(df['genre'].value_counts(dropna=False).sort_index())

df.to_csv(CSV_PATH, index=False)
print(f"\n💾 Сохранено: {CSV_PATH}")

=== После дополнительного объединения (19 уникальных) ===
genre
adventure            5
children             5
coming_of_age        2
contemporary         3
fantasy             19
gothic               9
historical           2
modernist            2
mystery_thriller     5
nonfiction           2
psychological        9
realism             12
romance              8
satire               6
scifi                6
short_story          1
social               7
tragedy              2
utopia               1
Name: count, dtype: int64

💾 Сохранено: /Users/kseniazavyalova/Desktop/meta_stats_of_book.csv


In [78]:
df['author_gender'].value_counts()

author_gender
female    55
male      51
Name: count, dtype: int64